In [1]:
# ============================================================
# USER MANUAL / QUICK ORIENTATION GUIDE
# ============================================================
# This notebook is designed for Google Colab.
# It mounts Google Drive, loads training and test CSV files from Drive,
# runs a fully nested machine-learning workflow, and saves all results
# back to Google Drive.
# TRAIN and TEST csv files should be saved from Excel in UTF-8 format. the first column is expected to be
# unlabeled and contain patient IDs. The second column should be labeled as "label" and contain a binary outcome.
# All other columns are features with feature names in the first row.
#
# -------------------------
# 1) MAJOR CHARACTERISTICS
# -------------------------
# - Environment:
#   This code is intended for Google Colab. Required Python packages are
#   installed inside the notebook, and Google Drive is mounted so that
#   input data and output results can be read/written directly.
#
# - Data location:
#   The code expects preprepared CSV files on Google Drive:
#       TRAIN_PATH = development / training dataset
#       TEST_PATH  = independent holdout test dataset
#   These files should contain:
#       first column = patient/sample ID
#       column "label" = binary outcome
#       remaining columns = numeric features
#
# - Fully nested validation:
#   The workflow uses a strict nested cross-validation design:
#       outer CV = unbiased out-of-fold validation
#       inner CV = hyperparameter selection
#   This helps reduce information leakage and gives more realistic
#   validation estimates than non-nested tuning.
#
# - Leakage-safe preprocessing:
#   All filtering, scaling, balancing, and feature selection steps are
#   performed inside the training folds of CV through the pipeline.
#
# - Consensus feature selection:
#   The code can use several feature selectors in parallel and then keep
#   features that receive enough votes across selectors.
#   This is controlled by USE_UNION_FS and related settings below.
#
# - Final refit and freezing:
#   After model comparison, the best configuration for each chosen
#   classifier can be refit on the full development set and saved as a
#   frozen .joblib model for later scoring of new patients.
#
#
# -------------------------
# 2) SETTINGS AVAILABLE
# -------------------------
# The main user-editable settings are grouped near the top of the notebook.
#
# A) Reproducibility and CV
#   RANDOM_SEED
#   OUTER_FOLDS / OUTER_REPEATS
#   INNER_FOLDS / INNER_REPEATS
#   SEARCH_NJOBS
#   N_ITER
#
# B) Prefiltering / preprocessing
#   BALANCING_METHOD      : "upsample", "smote", or "none"
#   SCALING_METHOD        : "zscore" or "minmax"
#   VARIANCE_THRESHOLD    : low-variance filter
#   USE_AUC_FILTER        : turn AUC-based univariate filter on/off
#   INVERT_LOW_AUC_FEATURES
#   MIN_FEATURES_AFTER_AUC
#   MAX_FEATURES_AFTER_AUC
#   USE_CORR_FILTER       : turn correlation filter on/off
#   GLOBAL_TUNE_GRID      : tuneable preprocessing thresholds such as:
#                           aucf__high, aucf__low, corr__threshold
#
# C) Feature selection
#   USE_UNION_FS          : enable/disable the consensus selector step
#   FS_METHODS_ALL        : all selector methods allowed
#   FS_COMBOS             : which selector combinations to test
#   UNION_FS_K            : how many features each selector proposes
#   UNION_FS_MIN_KEEP     : minimum number of features kept after consensus/top-up
#   UNION_FS_MIN_VOTES    : number of votes required for consensus
#
#   Available selector methods in this code:
#       N_MRMR
#       mRMR
#       N_BORUTA
#       N_L1
#       N_ENET
#       RFE
#
# D) PCA / speed
#   PCA_VARIANTS
#   PCA_VARIANCE_GRID
#   DROP_SLOW_MODELS
#
# E) Classifiers
#   RUN_CLASSIFIERS       : choose which classifiers to evaluate
#   SVM_KERNEL_MODE       : controls linear / RBF SVM behavior if SVM is used
#
#   The code includes implementations/grids for classifiers such as:
#       NAIVE_BAYES
#       GAUSSIAN_PROCESS
#       EXTRATREES
#       LOGREG
#       SGD_LOGLOSS
#       SVM
#       LDA
#       RANDOM_FOREST
#       DECISION_TREE
#       ADABOOST
#       XGBOOST
#       HIST_GBDT
#       LIGHTGBM
#       CATBOOST
#       BAGGING
#       AE (MLP)
#
# F) Optional post-hoc stage
#   USE_POSTHOC_IMPORTANCE_FS
#   If enabled, the code performs an additional post-hoc importance-based
#   feature selection / union / final refit stage after the main runs.
#
#
# -------------------------
# 3) INPUTS, OUTPUTS, AND WHERE RESULTS ARE SAVED
# -------------------------
# Google Drive is mounted here:
#       drive.mount("/content/drive", force_remount=True)
#
# Input sample files are read from:
#       TRAIN_PATH = "/content/drive/MyDrive/NESTED_ML/data/train.csv"
#       TEST_PATH  = "/content/drive/MyDrive/NESTED_ML/data/test.csv"
#
# Main output directory:
#       OUT_DIR = "/content/drive/MyDrive/NESTED_ML/results/ML.csv"
#
# For each tested model/configuration, the code creates:
#       OUT_DIR / model_id /
# and saves files such as:
#       fold_metrics.csv
#       fold_metrics_avg.csv
#       train_predictions.csv
#       val_oof_predictions.csv
#       test_predictions.csv
#       FSU_selector_outputs_long.csv
#       FSU_consensus_votes_long.csv
#       FSU_shap_meanabs_long.csv
#       FSU_union_features_summary.csv
#       meta.json
#
# Across all tested models, the code also saves:
#       OUT_DIR / all_models_summary_metrics.csv
#       OUT_DIR / all_param_combos_metrics.csv
#
# Frozen final single-classifier models are saved under:
#       OUT_DIR / FROZEN_SINGLE_MODELS /
# including:
#       *_frozen_model.joblib
#       frozen_models_summary.csv
#
# If post-hoc importance-based feature selection is enabled, additional
# post-hoc outputs are saved under the corresponding final output folder.
#
#
# -------------------------
# 4) PRACTICAL USAGE NOTES
# -------------------------
# - To change dataset locations, edit TRAIN_PATH, TEST_PATH, and OUT_DIR.
# - To change tested classifiers, edit RUN_CLASSIFIERS.
# - To change preprocessing, edit BALANCING_METHOD, SCALING_METHOD,
#   USE_AUC_FILTER, USE_CORR_FILTER, and GLOBAL_TUNE_GRID.
# - To change consensus feature selection, edit USE_UNION_FS, FS_COMBOS,
#   UNION_FS_K, UNION_FS_MIN_KEEP, and UNION_FS_MIN_VOTES.
# - To disable optional post-hoc importance selection, leave:
#       USE_POSTHOC_IMPORTANCE_FS = False
#
# The code assumes binary classification with a "label" column and uses
# the first column as the patient/sample identifier.
# ============================================================


import os
import json
import math
import shutil
import itertools
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Dict, Any
from collections import Counter

# Colab deps
!pip -q install imbalanced-learn
!pip -q install xgboost
!pip -q install lightgbm catboost
!pip -q install shap
import shap

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV, GridSearchCV, ParameterGrid

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.utils import check_random_state

from xgboost import XGBClassifier


from sklearn.metrics import (
    roc_auc_score, accuracy_score, balanced_accuracy_score,
    confusion_matrix, matthews_corrcoef, precision_score, recall_score, f1_score
)

from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier, BaggingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE


from scipy.stats import kruskal
from scipy.stats import loguniform, randint
from IPython.display import display

import lightgbm as lgb

import warnings
import time

import joblib
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

# ============================================================
# MASTER SWITCHES
# ============================================================
USE_UNION_FS = True                  # controls the in-pipeline feature selectors (fsu step)
USE_POSTHOC_IMPORTANCE_FS = False    # controls the big "POST-HOC FEATURE IMPORTANCE → FINAL REFIT" stage
USE_CORR_FILTER = True
USE_AUC_FILTER = True                # master switch for leakage-free univariate AUC filter


# ============================================================
# 1) USER SETTINGS (EDIT THESE)
# ============================================================

# -------------------------
# A) Reproducibility and CV
# -------------------------
RANDOM_SEED = 42

OUTER_FOLDS = 5
INNER_FOLDS = 5

OUTER_REPEATS = 1
INNER_REPEATS = 1

# Hyperparameter search budget per inner CV search
N_ITER = 30
SEARCH_NJOBS = 8


# -------------------------
# B) Prefiltering / preprocessing
# -------------------------
# Balancing: "upsample" or "smote" or "none"
BALANCING_METHOD = "upsample"

# Scaling: "zscore" or "minmax"
SCALING_METHOD = "zscore"

VARIANCE_THRESHOLD = 1e-6

# Univariate AUC feature filter (inside CV folds; leakage-free)
INVERT_LOW_AUC_FEATURES = True
MIN_FEATURES_AFTER_AUC = 50
MAX_FEATURES_AFTER_AUC = 3000   # can set: None

# Correlation filter is controlled by USE_CORR_FILTER above


# -------------------------
# C) Feature selection
# -------------------------
UNION_FS_K = 25
UNION_FS_MIN_KEEP = 4
UNION_FS_MIN_VOTES = 3

# Which selectors are allowed
FS_METHODS_ALL = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]

# Choose feature-selector combinations to run
FS_COMBOS = [
    ("N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET"),
]


def _validate_fs_combo(combo):
    combo = tuple(combo)
    bad = [m for m in combo if m not in FS_METHODS_ALL]
    if bad:
        raise ValueError(f"Unknown FS methods in combo {combo}: {bad}")
    if len(combo) < 1:
        raise ValueError("FS combo cannot be empty.")
    if UNION_FS_MIN_VOTES < 1 or UNION_FS_MIN_VOTES > len(combo):
        raise ValueError(
            f"UNION_FS_MIN_VOTES must be 1..{len(combo)} for combo {combo} "
            f"(got {UNION_FS_MIN_VOTES})"
        )
    return combo

FS_COMBOS = [_validate_fs_combo(c) for c in FS_COMBOS]
print(f"[UNION_FS] combos={len(FS_COMBOS)} | min_votes={UNION_FS_MIN_VOTES}")
print(f"[UNION_FS] selectors voting={len(FS_METHODS_ALL)} | min_votes={UNION_FS_MIN_VOTES}")


# -------------------------
# D) PCA / model-speed options
# -------------------------
PCA_VARIANTS = ["no_pca"]
PCA_VARIANCE_GRID = [0.90, 0.95, 0.99]

# Optional: drop slow models
DROP_SLOW_MODELS = False


# -------------------------
# E) Classifier choice
# -------------------------
# SVM kernels setting
SVM_KERNEL_MODE = "linear+rbf"   # "linear" or "linear+rbf"

# Choose classifiers to run
# RUN_CLASSIFIERS = ["RANDOM_FOREST", "BAGGING", "ADABOOST", "HIST_GBDT"]
RUN_CLASSIFIERS = [
    "NAIVE_BAYES",
    "GAUSSIAN_PROCESS",
    "EXTRATREES",
    "LOGREG",
    "SGD_LOGLOSS",
]

CLASSIFIERS = [c.upper() for c in RUN_CLASSIFIERS]
print("Configured CLASSIFIERS:", CLASSIFIERS)

_valid_classifiers = {
    "SVM", "LDA", "LOGREG", "ADABOOST", "RANDOM_FOREST", "DECISION_TREE",
    "XGBOOST", "NAIVE_BAYES", "GAUSSIAN_PROCESS", "AE",
    "EXTRATREES", "HIST_GBDT", "SGD_LOGLOSS", "LIGHTGBM", "CATBOOST", "BAGGING"
}

bad_c = [c for c in CLASSIFIERS if c not in _valid_classifiers]
if bad_c:
    raise ValueError(f"Unknown classifier(s) in RUN_CLASSIFIERS: {bad_c}")


# ============================================================
# 2) GLOBAL PREPROCESSING PARAM GRID TO TUNE
# ============================================================
GLOBAL_TUNE_GRID = {}

if USE_AUC_FILTER:
    GLOBAL_TUNE_GRID.update({
        "aucf__high": [0.58],
        "aucf__low":  [0.42],
    })

if USE_CORR_FILTER:
    GLOBAL_TUNE_GRID.update({
        "corr__threshold": [0.98],
    })

GLOBAL_COMBOS = list(ParameterGrid(GLOBAL_TUNE_GRID)) if GLOBAL_TUNE_GRID else [dict()]

print(
    f"[GLOBAL_GRID] combos={len(GLOBAL_COMBOS)} | "
    f"AUC_FILTER={USE_AUC_FILTER} | CORR_FILTER={USE_CORR_FILTER} | "
    f"keys={list(GLOBAL_TUNE_GRID.keys())}"
)


# ============================================================
# 3) GOOGLE DRIVE + FILE PATHS
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

TRAIN_PATH = "/content/drive/MyDrive/NESTED_ML/data/train.csv"
TEST_PATH  = "/content/drive/MyDrive/NESTED_ML/data/test.csv"

OUT_DIR = "/content/drive/MyDrive/NESTED_ML/results/EXP1"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# 4) LOAD DATA  (DEV=TRAIN, TEST=holdout)
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

id_col = train_df.columns[0]
label_col = "label"

assert label_col in train_df.columns, f"Expected label column '{label_col}' in train.csv"
assert label_col in test_df.columns,  f"Expected label column '{label_col}' in test.csv"

train_ids = train_df[id_col].astype(str).values
test_ids  = test_df[id_col].astype(str).values

y_train = train_df[label_col].values.astype(int)
y_test  = test_df[label_col].values.astype(int)

X_train = train_df.drop(columns=[id_col, label_col]).apply(pd.to_numeric, errors="coerce")
X_test  = test_df.drop(columns=[id_col, label_col]).apply(pd.to_numeric, errors="coerce")

feature_names = X_train.columns.tolist()

print("DEV (train) shape:", X_train.shape, " Pos%:", y_train.mean())
print("TEST shape:",        X_test.shape,  " Pos%:", y_test.mean())

# ============================================================
# 4) CUSTOM TRANSFORMERS (Leakage-free)
# ============================================================
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold
        self.keep_idx_ = None

    def fit(self, X, y=None):
        X = np.asarray(X)
        n_features = X.shape[1]
        if n_features <= 1:
            self.keep_idx_ = np.arange(n_features)
            return self

        variances = np.nanvar(X, axis=0)
        corr = np.corrcoef(X, rowvar=False)
        corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
        abs_corr = np.abs(corr)

        keep = np.ones(n_features, dtype=bool)

        def current_mean_abs_corr(mask):
            sub = abs_corr[np.ix_(mask, mask)]
            if sub.shape[0] <= 1:
                return np.zeros(sub.shape[0])
            return (sub.sum(axis=0) - 1.0) / (sub.shape[0] - 1)

        while True:
            idx = np.where(keep)[0]
            if len(idx) <= 1:
                break

            sub = abs_corr[np.ix_(keep, keep)].copy()
            np.fill_diagonal(sub, 0.0)

            max_val = sub.max()
            if max_val <= self.threshold:
                break

            a, b = np.argwhere(sub == max_val)[0]
            feat_a = idx[a]
            feat_b = idx[b]

            mean_corr = current_mean_abs_corr(keep)
            mean_a = mean_corr[a]
            mean_b = mean_corr[b]

            if mean_a > mean_b:
                drop = feat_a
            elif mean_b > mean_a:
                drop = feat_b
            else:
                drop = feat_a if variances[feat_a] < variances[feat_b] else feat_b

            keep[drop] = False

        self.keep_idx_ = np.where(keep)[0]
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        # assume upstream imputer handled NaNs; just be robust without using X-stats
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        return X[:, self.keep_idx_]



class UnivariateAUCFilterProper(BaseEstimator, TransformerMixin):
    def __init__(self, high=0.51, low=0.01, invert_low=True, min_keep=50, max_keep=None):
        self.high = float(high)
        self.low = float(low)
        self.invert_low = bool(invert_low)
        self.min_keep = int(min_keep)
        self.max_keep = None if max_keep is None else int(max_keep)

        self.keep_idx_ = None
        self.invert_mask_kept_ = None
        self.auc_ = None
        self.median_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y).astype(int)
        n, p = X.shape

        # train-only median fill (so transform on val/test is consistent)
        Xc = X.copy()
        Xc[~np.isfinite(Xc)] = np.nan
        med = np.nanmedian(Xc, axis=0)
        med = np.where(np.isfinite(med), med, 0.0)
        Xc = np.where(np.isfinite(Xc), Xc, med)
        self.median_ = med

        aucs = np.full(p, 0.5, dtype=float)

        # compute tie-correct AUC per feature
        for j in range(p):
            col = Xc[:, j]
            if np.nanstd(col) < 1e-12:
                continue
            try:
                aucs[j] = roc_auc_score(y, col)
            except Exception:
                aucs[j] = 0.5

        keep = (aucs >= self.high) | (aucs <= self.low)

        # fallback: ensure at least min_keep
        if keep.sum() < min(self.min_keep, p):
            k = min(self.min_keep, p)
            order = np.argsort(np.abs(aucs - 0.5))[::-1]
            keep = np.zeros(p, dtype=bool)
            keep[order[:k]] = True

        # --- NEW: cap to max_keep strongest features ---
        if self.max_keep is not None and keep.sum() > self.max_keep:
            kept_idx = np.where(keep)[0]
            strength = np.abs(aucs[kept_idx] - 0.5)   # bigger = more predictive
            order = kept_idx[np.argsort(strength)[::-1]]
            keep = np.zeros(p, dtype=bool)
            keep[order[: self.max_keep]] = True

        self.auc_ = aucs
        self.keep_idx_ = np.where(keep)[0]
        self.invert_mask_kept_ = (aucs[self.keep_idx_] <= self.low) if self.invert_low else np.zeros(len(self.keep_idx_), bool)
        return self


    def transform(self, X):
        X = np.asarray(X, dtype=float)
        Xc = X.copy()
        Xc[~np.isfinite(Xc)] = np.nan
        Xc = np.where(np.isfinite(Xc), Xc, self.median_)

        Xk = Xc[:, self.keep_idx_]
        if self.invert_low and self.invert_mask_kept_ is not None and self.invert_mask_kept_.any():
            Xk = Xk.copy()
            Xk[:, self.invert_mask_kept_] *= -1.0
        return Xk

def _safe_finite(X):
    X = np.asarray(X, dtype=float)
    X = np.where(np.isfinite(X), X, np.nan)
    # median fill per column
    med = np.nanmedian(X, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    X = np.where(np.isfinite(X), X, med)
    return X

def _topk_idx(scores, k):
    scores = np.asarray(scores, dtype=float)
    scores = np.nan_to_num(scores, nan=-np.inf, posinf=np.max(scores[np.isfinite(scores)]) if np.isfinite(scores).any() else 0.0, neginf=-np.inf)
    order = np.argsort(scores)[::-1]
    return order[:k]

def _rfe_topk(X, y, k, random_state=42):
    X = _safe_finite(X)
    y = np.asarray(y).astype(int)

    base = LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=1.0,
        max_iter=500,
        random_state=random_state,
    )
    rfe = RFE(estimator=base, n_features_to_select=min(k, X.shape[1]), step=0.2)
    rfe.fit(X, y)
    idx = np.where(rfe.get_support())[0]
    return idx[:k]

def _lasso_l1_topk(X, y, k, random_state=42):
    X = _safe_finite(X)
    y = np.asarray(y).astype(int)

    clf = LogisticRegression(
        solver="liblinear",
        penalty="l1",
        C=0.5,
        max_iter=1000,
        random_state=random_state,
    )
    clf.fit(X, y)
    coef = np.abs(clf.coef_).ravel()
    if coef.size == 0 or np.all(coef == 0):
        # fallback: variance
        coef = np.var(X, axis=0)
    return _topk_idx(coef, min(k, X.shape[1]))

def _enet_topk(X, y, k, random_state=42):
    # Use saga elasticnet logistic regression (stable for coefficients)
    X = _safe_finite(X)
    y = np.asarray(y).astype(int)

    clf = LogisticRegression(
        solver="saga",
        penalty="elasticnet",
        l1_ratio=0.5,
        C=1.0,
        max_iter=3000,
        random_state=random_state,
    )
    clf.fit(X, y)
    coef = np.abs(clf.coef_).ravel()
    if coef.size == 0 or np.all(coef == 0):
        coef = np.var(X, axis=0)
    return _topk_idx(coef, min(k, X.shape[1]))

def _mrmr_select(X, y, k, mode="diff", random_state=42):
    """
    Simple leakage-free mRMR inside-fold.
    mode:
      - "diff"  : MI - mean(|corr|)     (mRMR)
      - "ratio" : MI / (1 + mean(|corr|)) (N_MRMR-ish / normalized)
    """
    X = _safe_finite(X)
    y = np.asarray(y).astype(int)

    # Standardize for stable corr/MI behavior (kept local; doesn't leak)
    Xs = StandardScaler().fit_transform(X)

    mi = mutual_info_classif(Xs, y, random_state=random_state)
    mi = np.nan_to_num(mi, nan=0.0, posinf=0.0, neginf=0.0)

    # Precompute abs correlation (guarded)
    C = np.corrcoef(Xs, rowvar=False)
    C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
    absC = np.abs(C)
    np.fill_diagonal(absC, 0.0)

    p = Xs.shape[1]
    k = min(int(k), p)

    selected = []
    remaining = set(range(p))

    # start from highest MI
    first = int(np.argmax(mi)) if p > 0 else 0
    selected.append(first)
    remaining.discard(first)

    while len(selected) < k and len(remaining) > 0:
        sel_arr = np.array(selected, dtype=int)

        # redundancy = mean abs corr with already selected
        red = absC[:, sel_arr].mean(axis=1) if sel_arr.size > 0 else np.zeros(p)

        if mode == "ratio":
            score = mi / (1.0 + red)
        else:
            score = mi - red

        # prevent re-picking selected
        score[sel_arr] = -np.inf
        nxt = int(np.argmax(score))
        if not np.isfinite(score[nxt]):
            # fallback: pick best MI among remaining
            rem = np.array(list(remaining), dtype=int)
            nxt = int(rem[np.argmax(mi[rem])])
        selected.append(nxt)
        remaining.discard(nxt)

    return np.array(selected[:k], dtype=int)

def _boruta_lite_topk(X, y, k, random_state=42, n_estimators=150):
    """
    Lightweight Boruta-like:
      - create shadow features by column-wise shuffling
      - fit RF, compare importances to max shadow
      - if too few survive, top-up by importance
    """
    X = _safe_finite(X)
    y = np.asarray(y).astype(int)

    n, p = X.shape
    k = min(int(k), p)

    rng = np.random.RandomState(random_state)
    X_shadow = X.copy()
    for j in range(p):
        rng.shuffle(X_shadow[:, j])

    Xb = np.hstack([X, X_shadow])

    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=1,
        class_weight="balanced_subsample",
    )
    rf.fit(Xb, y)
    imp = np.asarray(rf.feature_importances_, dtype=float)

    imp_real = imp[:p]
    imp_shadow = imp[p:]
    thr = float(np.max(imp_shadow)) if imp_shadow.size else 0.0

    keep = np.where(imp_real > thr)[0]
    if keep.size >= k:
        # choose strongest among those
        keep = keep[np.argsort(imp_real[keep])[::-1][:k]]
        return keep.astype(int)

    # top-up by raw importance
    order = np.argsort(imp_real)[::-1]
    keep2 = np.unique(np.concatenate([keep, order[:k]]))
    # ensure size k (deterministic)
    keep2 = keep2[np.argsort(imp_real[keep2])[::-1]]
    return keep2[:k].astype(int)


class UnionFeatureSelector50(BaseEstimator, TransformerMixin):
    def __init__(self, k=50, min_keep=50, min_votes=2, random_state=42, methods=("N_MRMR","mRMR","N_BORUTA","N_L1","N_ENET","RFE")):
        self.k = int(k)
        self.min_keep = int(min_keep)
        self.min_votes = int(min_votes)
        self.random_state = int(random_state)
        self.methods = tuple(methods)  # enabled selector methods
        self.keep_idx_ = None
        self.details_ = None

    def fit(self, X, y):
        X = _safe_finite(X)
        y = np.asarray(y).astype(int)

        p = X.shape[1]
        k = min(self.k, p)

        methods = tuple(self.methods)
        if len(methods) < 1:
            raise ValueError("UnionFeatureSelector50.methods cannot be empty.")
        if not (1 <= self.min_votes <= len(methods)):
            raise ValueError(f"min_votes must be 1..{len(methods)} (got {self.min_votes})")

        # Run only enabled selectors
        selector_sets = {}

        if "RFE" in methods:
            idx_rfe = _rfe_topk(X, y, k, random_state=self.random_state)
            selector_sets["RFE"] = set(map(int, idx_rfe))
        else:
            idx_rfe = np.array([], dtype=int)

        if "N_MRMR" in methods:
            idx_nmrmr = _mrmr_select(X, y, k, mode="ratio", random_state=self.random_state)
            selector_sets["N_MRMR"] = set(map(int, idx_nmrmr))
        else:
            idx_nmrmr = np.array([], dtype=int)

        if "mRMR" in methods:
            idx_mrmr = _mrmr_select(X, y, k, mode="diff", random_state=self.random_state)
            selector_sets["mRMR"] = set(map(int, idx_mrmr))
        else:
            idx_mrmr = np.array([], dtype=int)

        if "N_BORUTA" in methods:
            idx_bor = _boruta_lite_topk(X, y, k, random_state=self.random_state)
            selector_sets["N_BORUTA"] = set(map(int, idx_bor))
        else:
            idx_bor = np.array([], dtype=int)

        if "N_L1" in methods:
            idx_l1 = _lasso_l1_topk(X, y, k, random_state=self.random_state)
            selector_sets["N_L1"] = set(map(int, idx_l1))
        else:
            idx_l1 = np.array([], dtype=int)

        if "N_ENET" in methods:
            idx_enet = _enet_topk(X, y, k, random_state=self.random_state)
            selector_sets["N_ENET"] = set(map(int, idx_enet))
        else:
            idx_enet = np.array([], dtype=int)

        # ============================================================
        # CONSENSUS FS (>=min_votes) across *enabled* selectors
        # ============================================================
        vote_counter = Counter()
        for s in selector_sets.values():
            vote_counter.update(s)

        mv = int(self.min_votes)
        consensus = np.array(sorted([i for i, c in vote_counter.items() if c >= mv]), dtype=int)

        # Top-up if consensus too small
        if consensus.size < min(self.min_keep, p):
            ranked_by_votes = [i for i, c in vote_counter.most_common()]
            kept = list(consensus.tolist())

            for i in ranked_by_votes:
                if i not in kept:
                    kept.append(int(i))
                if len(kept) >= min(self.min_keep, p):
                    break

            if len(kept) < min(self.min_keep, p):
                Xs = StandardScaler().fit_transform(X)
                mi = mutual_info_classif(Xs, y, random_state=self.random_state)
                topup = _topk_idx(mi, min(self.min_keep, p))
                for i in topup.tolist():
                    if int(i) not in kept:
                        kept.append(int(i))
                    if len(kept) >= min(self.min_keep, p):
                        break

            consensus = np.array(sorted(set(kept)), dtype=int)

        self.keep_idx_ = consensus

        vote_dict = {int(i): int(vote_counter[i]) for i in consensus.tolist()}
        # details_ now only reports what was enabled
        self.details_ = {
            "METHODS": list(methods),
            "RFE": idx_rfe.tolist() if idx_rfe.size else [],
            "N_MRMR": idx_nmrmr.tolist() if idx_nmrmr.size else [],
            "mRMR": idx_mrmr.tolist() if idx_mrmr.size else [],
            "N_BORUTA": idx_bor.tolist() if idx_bor.size else [],
            "N_L1": idx_l1.tolist() if idx_l1.size else [],
            "N_ENET": idx_enet.tolist() if idx_enet.size else [],
            "CONSENSUS_MIN_VOTES": int(self.min_votes),
            "CONSENSUS_N": int(len(consensus)),
            "CONSENSUS_VOTES": vote_dict,
        }
        return self


    def transform(self, X):
        X = np.asarray(X, dtype=float)
        # Upstream SimpleImputer should already handle NaNs; keep it leak-safe:
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        return X[:, self.keep_idx_]


# ============================================================
# 5) METRICS + THRESHOLDING
# ============================================================
def optimal_threshold_youden(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float).ravel()

    # define scores safely
    scores = np.unique(y_score[np.isfinite(y_score)])
    if scores.size == 0:
        return 0.0

    # If too many unique scores, use quantiles to keep it fast
    if scores.size > 1000:
        scores = np.quantile(y_score, np.linspace(0, 1, 1001))
        scores = np.sort(np.unique(scores))

    # Evaluate midpoints between consecutive unique scores (more correct)
    if scores.size >= 2:
        thresholds = (scores[:-1] + scores[1:]) / 2.0
        thresholds = np.r_[scores[0] - 1e-12, thresholds, scores[-1] + 1e-12]
    else:
        thresholds = np.array([scores[0]])

    best_t = float(thresholds[0])
    best_j = -1e9

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        j = sens + spec - 1.0

        if j > best_j:
            best_j = j
            best_t = float(t)

    return best_t


def compute_metrics(y_true, y_score, threshold=None):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan

    if threshold is None:
        threshold = optimal_threshold_youden(y_true, y_score)

    y_pred = (y_score >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    youden = sens + spec - 1.0

    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1v  = f1_score(y_true, y_pred, zero_division=0)

    return {
        "AUC": float(auc),
        "ACC": float(acc),
        "BACC": float(bacc),
        "MCC": float(mcc),
        "SENS": float(sens),
        "SPEC": float(spec),
        "YOUDEN": float(youden),
        "PREC": float(prec),
        "RECALL": float(rec),
        "F1": float(f1v),
        "THRESH": float(threshold),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

def predict_score(estimator, X):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        return np.asarray(s).ravel()
    return estimator.predict(X).astype(float)

def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true) if y_true is not None else None
    y_score = np.asarray(y_score)
    if y_true is None:
        return np.nan
    if len(np.unique(y_true)) != 2:
        return np.nan
    try:
        return float(roc_auc_score(y_true.astype(int), y_score.astype(float)))
    except Exception:
        return np.nan


# ============================================================
# 6) BUILD COMPONENTS
# ============================================================
def make_sampler(y_for_fold=None):
    m = (BALANCING_METHOD or "none").lower().strip()
    if m == "upsample":
        return RandomOverSampler(random_state=RANDOM_SEED)
    if m == "smote":
        # if provided, adapt k to the fold’s minority class count
        k = 5
        if y_for_fold is not None:
            y = np.asarray(y_for_fold).astype(int)
            vals, counts = np.unique(y, return_counts=True)
            if len(counts) == 2:
                minority = int(np.min(counts))
                k = max(1, min(5, minority - 1))
        return SMOTE(random_state=RANDOM_SEED, k_neighbors=k)
    if m in ("none", "off", "no", ""):
        return "passthrough"
    raise ValueError("BALANCING_METHOD must be 'upsample', 'smote', or 'none'")



def make_scaler():
    if SCALING_METHOD.lower() == "zscore":
        return StandardScaler()

    elif SCALING_METHOD.lower() == "minmax":
        return MinMaxScaler()

    else:
        raise ValueError("SCALING_METHOD must be 'zscore' or 'minmax'")


# ============================================================
# STRICT classifier + selector grids (NO EXTRA hyperparams/values)
# Matches exactly the user's specification.
# ============================================================

from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# ---- OPTIONAL: if you already have an MRMRSelector class, keep this.
# If your mRMR implementation is elsewhere, just ensure selector_and_grid("mrmr") returns it.
# class MRMRSelector(...): ...


def classifier_and_grid(clf_name: str):
    """
    STRICT classifier grids: only what's explicitly allowed.
    """
    clf_name = clf_name.lower()

    # 1) AdaBoost - EXACT
    if clf_name == "adaboost":
        clf = AdaBoostClassifier(random_state=RANDOM_SEED)
        dist = {"clf__n_estimators": [10, 50, 100]}
        return clf, dist

    # 2) AE (MLP) - EXACT
    if clf_name == "ae":
        clf = MLPClassifier(solver="adam", random_state=RANDOM_SEED)
        dist = {
            "clf__hidden_layer_sizes": [(30,), (100,)],
            "clf__alpha": [0.0001, 0.001],
            "clf__learning_rate_init": [0.001, 0.01],
        }
        return clf, dist

    # 3) Decision Tree - EXACT
    if clf_name == "decision_tree":
        clf = DecisionTreeClassifier(random_state=RANDOM_SEED)
        dist = {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_features": ["log2", "sqrt"],
        }
        return clf, dist

    # 4) Gaussian Process - EXACT (no tuning)
    if clf_name == "gaussian_process":
        clf = GaussianProcessClassifier()
        dist = {}
        return clf, dist


    # 5) LDA - EXACT (3 configs)
    if clf_name == "lda":
        clf = LinearDiscriminantAnalysis()
        dist = [
            {"clf__solver": ["svd"]},
            {"clf__solver": ["lsqr"], "clf__shrinkage": ["auto"]},
            {"clf__solver": ["eigen"], "clf__shrinkage": ["auto"]},
        ]
        return clf, dist

    # 6) Logistic Regression - EXACT
    if clf_name == "logreg":
        clf = LogisticRegression(solver="liblinear", random_state=RANDOM_SEED)
        dist = {
            "clf__tol": [0.00003, 0.0003, 0.003],
            "clf__C": [0.01, 0.1, 1.0, 5],
            "clf__max_iter": [100, 200],
        }
        return clf, dist

    # 7) LASSO classifier (LogReg L1) - EXACT
    if clf_name == "lasso":
        clf = LogisticRegression(
            solver="liblinear", penalty="l1", random_state=RANDOM_SEED
        )
        dist = {
            "clf__C": [0.1, 1.0],
            "clf__tol": [0.00003, 0.0003],
            "clf__max_iter": [100],
        }
        return clf, dist


    # 8) Random Forest - EXACT
    if clf_name == "random_forest":
        clf = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)
        dist = {
            "clf__n_estimators": [10, 50, 100],
            "clf__criterion": ["gini", "entropy"],
        }
        return clf, dist


    # 9) SVM - kernels controlled by SVM_KERNEL_MODE
    if clf_name == "svm":
        clf = SVC(probability=False, random_state=RANDOM_SEED)

        mode = (SVM_KERNEL_MODE or "linear").lower().strip()

        if mode == "linear":
            dist = [{"clf__kernel": ["linear"], "clf__C": [0.1, 0.3, 1.0, 3.0]}]

        elif mode in ("linear+rbf", "rbf+linear"):
            dist = [
                {"clf__kernel": ["linear"], "clf__C": [0.1, 0.3, 1.0, 3.0]},
                {"clf__kernel": ["rbf"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__gamma": [0.1, 0.3, 1.0, 3.0]},
            ]

        elif mode == "rbf":
            dist = [{"clf__kernel": ["rbf"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__gamma": [0.1, 0.3, 1.0, 3.0]}]

        elif mode == "poly":
            dist = [{"clf__kernel": ["poly"], "clf__C": [0.1, 0.3, 1.0, 3.0], "clf__degree": [2, 3, 4]}]

        else:
            raise ValueError(
                f"Unknown SVM_KERNEL_MODE='{SVM_KERNEL_MODE}'. Use 'linear', 'rbf', 'poly', or 'linear+rbf'."
            )

        return clf, dist


    # 10) XGBoost - (strict-ish, small grid)
    if clf_name == "xgboost":
        clf = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=1,
            tree_method="hist",
        )
        dist = {
            "clf__n_estimators": [50, 150],
            "clf__max_depth": [2, 3, 4],
            "clf__learning_rate": [0.03, 0.1],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.6, 0.8, 1.0],
            "clf__min_child_weight": [1, 5],
            "clf__reg_lambda": [1.0, 5.0],
        }
        return clf, dist

    # 11) Naive Bayes (GaussianNB) - no tuning (strict)

    if clf_name == "naive_bayes":
        clf = GaussianNB()
        dist = {}
        return clf, dist


    # 12) ExtraTrees - fast & strong on tabular
    if clf_name == "extratrees":
        clf = ExtraTreesClassifier(
            random_state=RANDOM_SEED,
            n_jobs=1,
        )
        dist = {
            "clf__n_estimators": [200, 600],
            "clf__max_depth": [None, 3, 5],
            "clf__min_samples_leaf": [1, 2, 5],
            "clf__max_features": ["sqrt", "log2", 1.0],
        }
        return clf, dist

    # 13) HistGradientBoosting (sklearn) - very efficient
    if clf_name in ("hist_gbdt", "histgradientboosting", "histgradientboostingclassifier"):
        clf = HistGradientBoostingClassifier(random_state=RANDOM_SEED)
        dist = {
            "clf__learning_rate": [0.03, 0.1],
            "clf__max_depth": [2, 3, None],
            "clf__max_leaf_nodes": [15, 31, 63],
            "clf__min_samples_leaf": [5, 10, 20],
            "clf__l2_regularization": [0.0, 0.1, 1.0],
        }
        return clf, dist

    # 14) SGDClassifier (log loss) - strong linear baseline, very fast
    if clf_name in ("sgd_logloss", "sgd"):
        clf = SGDClassifier(
            loss="log_loss",
            penalty="elasticnet",
            random_state=RANDOM_SEED,
            max_iter=3000,
            tol=1e-3,
        )
        dist = {
            "clf__alpha": [1e-5, 1e-4, 1e-3],
            "clf__l1_ratio": [0.0, 0.15, 0.5, 0.85],
            "clf__class_weight": [None, "balanced"],
        }
        return clf, dist

    # 15) LightGBM - often excellent on tabular
    if clf_name in ("lightgbm", "lgbm"):
        clf = LGBMClassifier(
            objective="binary",
            random_state=RANDOM_SEED,
            n_jobs=1,
            verbose=-1,
            verbosity=-1,
        )
        dist = {
            "clf__n_estimators": [200, 600],
            "clf__learning_rate": [0.03, 0.1],
            "clf__num_leaves": [7, 15, 31],
            "clf__min_child_samples": [5, 10, 20],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.8, 1.0],
            "clf__reg_lambda": [0.0, 1.0, 5.0],
        }
        return clf, dist

    # 16) CatBoost - strong, robust defaults; keep it quiet in Colab
    if clf_name == "catboost":
        clf = CatBoostClassifier(
            loss_function="Logloss",
            random_seed=RANDOM_SEED,
            verbose=0,
            allow_writing_files=False,
            thread_count=1,
        )
        dist = {
            "clf__iterations": [300, 800],
            "clf__depth": [2, 3, 4, 5],
            "clf__learning_rate": [0.03, 0.1],
            "clf__l2_leaf_reg": [1.0, 3.0, 10.0],
        }
        return clf, dist

    # 17) Bagging - stabilizes noisy small datasets
    if clf_name == "bagging":
        base = DecisionTreeClassifier(random_state=RANDOM_SEED)
        try:
            clf = BaggingClassifier(
                estimator=base,
                random_state=RANDOM_SEED,
                n_jobs=1,
            )
            est_key = "clf__estimator__"
        except TypeError:
            clf = BaggingClassifier(
                base_estimator=base,
                random_state=RANDOM_SEED,
                n_jobs=1,
            )
            est_key = "clf__base_estimator__"

        dist = {
            "clf__n_estimators": [50, 200, 500],
            "clf__max_samples": [0.7, 1.0],
            "clf__max_features": [0.7, 1.0],
            "clf__bootstrap": [True, False],
            f"{est_key}max_depth": [None, 2, 3, 4],
            f"{est_key}min_samples_leaf": [1, 2, 5],
        }
        return clf, dist

    raise ValueError(f"Unknown classifier '{clf_name}' (strict mode).")


# ============================================================
# 7) PIPELINE  (MODIFY build_pipeline to INCLUDE GLOBAL_TUNE_GRID)
# ============================================================
def build_pipeline(pca_variant: str, selector_name: str, clf_name: str, fs_methods=None):

    imputer = SimpleImputer(strategy="median")
    var = VarianceThreshold(threshold=VARIANCE_THRESHOLD)

    aucf = (

        UnivariateAUCFilterProper(
            high=0.5,   # placeholder – will be overridden by aucf__high
            low=0.5,    # placeholder – will be overridden by aucf__low
            invert_low=INVERT_LOW_AUC_FEATURES,
            min_keep=MIN_FEATURES_AFTER_AUC,
            max_keep=MAX_FEATURES_AFTER_AUC,
        )
        if USE_AUC_FILTER
        else "passthrough"
    )

    scaler = make_scaler()

    # Correlation filter (switchable; threshold overridden by gparams if tuned)
    corr = CorrelationFilter(threshold=0.95) if USE_CORR_FILTER else "passthrough"

    if fs_methods is None:
        fs_methods = tuple(FS_METHODS_ALL)

    fsu = (
        UnionFeatureSelector50(
            k=UNION_FS_K,
            min_keep=UNION_FS_MIN_KEEP,
            min_votes=min(UNION_FS_MIN_VOTES, len(fs_methods)),  # safety
            random_state=RANDOM_SEED,
            methods=tuple(fs_methods),
        )
        if USE_UNION_FS
        else "passthrough"
    )

    if pca_variant == "pca":
        pca = PCA(svd_solver="full")
    else:
        pca = "passthrough"

    clf, clf_dist = classifier_and_grid(clf_name)
    sampler = make_sampler()

    pipe = ImbPipeline(steps=[
        ("impute", imputer),
        ("var", var),
        ("aucf", aucf),
        ("scale", scaler),
        ("corr", corr),
        ("fsu", fsu),
        ("sample", sampler),
        ("pca", pca),
        ("clf", clf),
    ])



    # ---- Param grid ----
    if pca_variant == "pca":
        if isinstance(clf_dist, list):
            dist = []
            for d in clf_dist:
                merged = dict(d)
                merged.update({"pca__n_components": PCA_VARIANCE_GRID})
                dist.append(merged)
        else:
            dist = dict(clf_dist)
            dist.update({"pca__n_components": PCA_VARIANCE_GRID})
        return pipe, dist

    # no_pca
    if isinstance(clf_dist, list):
        dist = []
        for d in clf_dist:
            merged = dict(d)
            dist.append(merged)
        return pipe, dist

    dist = dict(clf_dist) if clf_dist else {}
    return pipe, dist


def make_search(estimator, param_grid, inner_cv):
    if not param_grid:
        return None

    return GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        error_score="raise",
        scoring="roc_auc",
        cv=inner_cv,
        n_jobs=SEARCH_NJOBS,
        refit=True,
        verbose=0
    )


def _step_support_indices(step, n_features):
    if hasattr(step, "get_support"):
        mask = step.get_support()
        return np.where(mask)[0]

    for attr in ["top_idx_", "selected_idx_", "support_", "keep_idx_"]:
        if hasattr(step, attr) and getattr(step, attr) is not None:
            idx = np.asarray(getattr(step, attr))
            if idx.dtype == bool:
                return np.where(idx)[0]
            return idx.astype(int)

    return np.arange(n_features)

def extract_selected_feature_names(fitted_pipe, feature_names):
    names = list(feature_names)

    if hasattr(fitted_pipe, "named_steps"):
        steps = list(fitted_pipe.named_steps.items())
    elif hasattr(fitted_pipe, "steps"):
        steps = list(fitted_pipe.steps)
    else:
        return names

    for name, step in steps:
        if step is None:
            continue
        if isinstance(step, str) and step == "passthrough":
            continue

        if name.lower() in ["sample","sampler","balance","scaler","normalize","norm","scale","impute","clf"]:
            continue

        if "pca" in name.lower():
            n_comp = getattr(step, "n_components_", None)
            if n_comp is None:
                n_comp = getattr(step, "n_components", None)
            if n_comp is None:
                continue
            try:
                n_comp_int = int(n_comp)
            except Exception:
                n_comp_int = int(getattr(step, "n_components_", len(names)))
            names = [f"PC{i+1}" for i in range(n_comp_int)]
            continue

        idx = _step_support_indices(step, len(names))
        idx = idx[(idx >= 0) & (idx < len(names))]
        names = [names[i] for i in idx]

    return names

def feature_names_before_step(fitted_pipe, feature_names, stop_step="fsu"):
    """
    Returns the feature-name list *entering* stop_step (e.g., entering 'fsu').
    This is crucial because fsu.details_ indices are relative to its input space
    (after var/aucf/corr etc).
    """
    names = list(feature_names)

    for name, step in fitted_pipe.steps:
        lname = name.lower()

        if lname == stop_step.lower():
            break

        if step is None or (isinstance(step, str) and step == "passthrough"):
            continue

        if lname in ["sample", "sampler"]:
            break
        if lname == stop_step.lower():
            break

        # skip steps that don't change feature dimension in a name-traceable way
        if lname in ["impute", "scale", "scaler", "normalize", "norm"]:
            continue

        if "pca" in lname:
            n_comp = getattr(step, "n_components_", getattr(step, "n_components", None))
            if n_comp is None:
                continue
            try:
                n_comp = int(n_comp)
            except Exception:
                n_comp = len(names)
            names = [f"PC{i+1}" for i in range(n_comp)]
            continue

        idx = _step_support_indices(step, len(names))
        idx = np.asarray(idx, dtype=int)
        idx = idx[(idx >= 0) & (idx < len(names))]
        names = [names[i] for i in idx]

    return names


def fsu_details_to_fold_df(best_est, orig_feature_names, fold_id, model_id):
    """
    Builds a per-fold DataFrame of ALL features selected by ANY selector inside fsu,
    including selector votes.
    """
    fsu = best_est.named_steps.get("fsu", None)
    if fsu is None or (isinstance(fsu, str) and fsu == "passthrough"):
        return pd.DataFrame()

    det = getattr(fsu, "details_", None)
    if not isinstance(det, dict):
        return pd.DataFrame()

    in_names = feature_names_before_step(best_est, orig_feature_names, stop_step="fsu")

    selector_keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]
    picked_by = {}

    for sk in selector_keys:
        idx_list = det.get(sk, [])
        if not idx_list:
            continue
        for i in idx_list:
            i = int(i)
            if 0 <= i < len(in_names):
                fn = in_names[i]
                picked_by.setdefault(fn, set()).add("RFE" if sk == "RFE" else sk)

    kept_idx = getattr(fsu, "keep_idx_", None)
    kept_feats = set()
    if kept_idx is not None:
        for i in np.asarray(kept_idx, dtype=int).tolist():
            if 0 <= i < len(in_names):
                kept_feats.add(in_names[i])

    rows = []
    for feat, sels in picked_by.items():
        rows.append({
            "model_id": model_id,
            "fold": int(fold_id),
            "feature": feat,
            "selector_votes": int(len(sels)),
            "selectors": "|".join(sorted(sels)),
            "kept_by_consensus": bool(feat in kept_feats),
            "coef_or_importance": np.nan,
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["kept_by_consensus", "selector_votes", "feature"],
                            ascending=[False, False, True]).reset_index(drop=True)
    return df



def _transform_until_clf(fitted_pipe, X_df):
    """
    Transform X_df through pipeline steps up to (but excluding) sampler and classifier.
    Returns:
      Xmat: numpy array in classifier-input feature space
      feat_names: feature names aligned to Xmat columns (best-effort)
    """
    # Feature names entering classifier (stop at 'clf' but break at 'sample' in name-trace)
    feat_names = feature_names_before_step(fitted_pipe, X_df.columns.tolist(), stop_step="clf")

    Xcur = X_df
    for name, step in fitted_pipe.steps:
        lname = name.lower()

        # stop before sampler/classifier
        if lname in ("sample", "sampler", "clf"):
            break

        if step is None or (isinstance(step, str) and step == "passthrough"):
            continue

        # Important: after first transform, Xcur may become np.array
        Xcur = step.transform(Xcur)

    Xmat = np.asarray(Xcur)
    return Xmat, feat_names


def jaccard_mean(list_of_sets):
    if len(list_of_sets) < 2:
        return 1.0
    pairs = list(itertools.combinations(list_of_sets, 2))
    vals = []
    for a, b in pairs:
        if len(a) == 0 and len(b) == 0:
            vals.append(1.0)
        else:
            vals.append(len(a & b) / max(1, len(a | b)))
    return float(np.mean(vals))

def save_stability_csv(model_tag, selected_sets, out_dir):
    counts = Counter([f for s in selected_sets for f in s])
    rows = []
    k = len(selected_sets)
    for feat, c in counts.most_common():
        rows.append([feat, c, c / max(1, k)])
    df = pd.DataFrame(rows, columns=["feature", "count", "freq"])
    df.to_csv(os.path.join(out_dir, f"{model_tag}__feature_stability.csv"), index=False)

def _print_end_summary(title: str, lines: list):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    for ln in lines:
        print(ln)
    print("=" * 90 + "\n")

    """
    Prints a single end-of-run sanity block:
      - mean sanity counts across folds (var/aucf/corr/fsu/pca)
      - mean selector output sizes across folds (N_MRMR, mRMR, ...)
      - post-hoc classifier-selected features (union/consensus/final passed)
      - final refit classifiers feature count
    """
    lines = []

    # ---- 1) fold-level preprocessing counts ----
    if "sanity_counts_json" in results_df.columns and results_df["sanity_counts_json"].notna().any():
        all_rows = []
        for s in results_df["sanity_counts_json"].dropna().tolist():
            try:
                lst = json.loads(s) if isinstance(s, str) else s
                if isinstance(lst, list):
                    all_rows.extend(lst)
            except Exception:
                pass

        if all_rows:
            df = pd.DataFrame(all_rows)
            for k in ["var", "aucf", "corr", "fsu", "pca"]:
                if k in df.columns:
                    lines.append(f"[SANITY-OVERALL] mean n_features after {k}: {np.nanmean(df[k].astype(float)):.2f}")

    # ---- 2) fold-level selector outputs ----
    if "fsu_counts_json" in results_df.columns and results_df["fsu_counts_json"].notna().any():
        all_rows = []
        for s in results_df["fsu_counts_json"].dropna().tolist():
            try:
                lst = json.loads(s) if isinstance(s, str) else s
                if isinstance(lst, list):
                    all_rows.extend(lst)
            except Exception:
                pass

        if all_rows:
            df = pd.DataFrame(all_rows)
            keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE", "CONSENSUS_KEPT"]
            present = [k for k in keys if k in df.columns]
            if present:
                parts = [f"{k}={np.nanmean(df[k].astype(float)):.2f}" for k in present]
                lines.append("[FSU-OVERALL] mean features selected per fold: " + " | ".join(parts))

    # ---- 3) post-hoc classifier feature selection ----
    if posthoc_union_count is not None:
        lines.append(f"[POSTHOC] classifier-selected UNION features (unique across top models): {int(posthoc_union_count)}")
    if posthoc_consensus_count is not None:
        lines.append(f"[POSTHOC] classifier-selected CONSENSUS features (vote rule): {int(posthoc_consensus_count)}")
    if final_feats_count is not None:
        lines.append(f"[POSTHOC] features PASSED to final refit classifiers: {int(final_feats_count)}")

    # ---- 4) final refit classifiers ----
    if final_refit_classifiers:
        lines.append(f"[FINAL REFIT] classifiers: {', '.join(final_refit_classifiers)}")

    _print_end_summary("OVERALL SANITY CHECK", lines)


def compute_fold_mean_abs_shap(best_est, X_tr_df, y_tr, max_samples=200, perm_repeats=5):
    """
    Returns list of dicts: [{"feature":..., "mean_abs_shap":...}, ...]
    Preference:
      1) SHAP (TreeExplainer / LinearExplainer) on TRAIN fold only
      2) If SHAP not available, leakage-safe PERMUTATION IMPORTANCE on TRAIN fold only,
         stored in the same key 'mean_abs_shap' (so it writes into the same column).
    Notes:
      - Works on RAW input features (X_tr_df columns).
      - Perm importance is computed on a subsample for speed.
      - No val/test touched => leakage-safe.
    """
    clf = best_est.named_steps.get("clf", None)
    if clf is None:
        return []

    # --- subsample TRAIN fold for speed ---
    Xuse = X_tr_df
    yuse = np.asarray(y_tr).astype(int)

    n = len(yuse)
    if n == 0:
        return []

    if n > max_samples:
        rng = check_random_state(42)
        idx = rng.choice(n, size=max_samples, replace=False)
        Xuse = X_tr_df.iloc[idx].copy()
        yuse = yuse[idx]

    # ============================================================
    # 1) Try SHAP first (fast path: tree + linear)
    # ============================================================
    try:
        # IMPORTANT: SHAP needs classifier-input space
        Xmat, feat_names = _transform_until_clf(best_est, Xuse)

        model_name = clf.__class__.__name__.lower()
        is_tree = hasattr(clf, "feature_importances_") or any(k in model_name for k in
                    ["forest", "tree", "xgb", "lgbm", "catboost", "histgradient"])
        is_linear = hasattr(clf, "coef_") and any(k in model_name for k in ["logistic", "sgd"])

        if is_tree:
            explainer = shap.TreeExplainer(clf)
            sv = explainer.shap_values(Xmat)
        elif is_linear:
            explainer = shap.LinearExplainer(clf, Xmat)
            sv = explainer.shap_values(Xmat)
        else:
            sv = None

        if sv is not None:
            # shap may return list for binary; take class-1 if needed
            if isinstance(sv, list) and len(sv) >= 2:
                sv = sv[1]
            sv = np.asarray(sv)
            if sv.ndim == 2:
                mean_abs = np.mean(np.abs(sv), axis=0)
                m = min(len(mean_abs), len(feat_names))
                mean_abs = mean_abs[:m]
                feat_names = feat_names[:m]
                return [{"feature": str(feat_names[i]), "mean_abs_shap": float(mean_abs[i])} for i in range(m)]
    except Exception:
        pass

    # ============================================================
    # 2) Fallback: permutation importance on TRAIN fold only (RAW features)
    # ============================================================
    try:
        # permutation_importance will permute RAW columns of Xuse
        # best_est is already fit on X_tr fold before you call this
        r = permutation_importance(
            best_est,
            Xuse,
            yuse,
            scoring="roc_auc",
            n_repeats=int(perm_repeats),
            random_state=42,
            n_jobs=1
        )

        imp = np.asarray(r.importances_mean, dtype=float)
        feat_names = list(Xuse.columns)

        # make safe
        m = min(len(imp), len(feat_names))
        imp = imp[:m]
        feat_names = feat_names[:m]

        # store in the same key name so it ends up in mean_abs_shap column
        return [{"feature": str(feat_names[i]), "mean_abs_shap": float(imp[i])} for i in range(m)]
    except Exception:
        return []
# ============================================================
# SANITY: feature counts per preprocessing step
# ============================================================
def feature_counts_by_step(fitted_pipe, X_df, steps=("var","aucf","corr","fsu","pca")):
    """
    Returns dict: {step_name: n_features_after_step}
    """
    Xcur = X_df
    out = {}

    for name, step in fitted_pipe.steps:
        lname = name.lower()

        if lname in ("sample", "sampler", "clf"):
            break

        if step is None or (isinstance(step, str) and step == "passthrough"):
            continue

        Xcur = step.transform(Xcur)

        if lname in steps:
            out[lname] = int(np.asarray(Xcur).shape[1])

    for k in steps:
        out.setdefault(k, None)

    return out


def pca_n_components_from_est(estimator):
    pca = estimator.named_steps.get("pca", None)
    if pca is None or isinstance(pca, str) or pca == "passthrough":
        return None
    return getattr(pca, "n_components_", getattr(pca, "n_components", None))


# ============================================================
# 8) NESTED TRAINING/EVALUATION
# ============================================================
@dataclass
class ModelRunResult:
    model_id: str
    pca_variant: str
    selector: str
    classifier: str
    best_params: Dict[str, Any]
    train_metrics: Dict[str, Any]
    val_metrics: Dict[str, Any]
    test_metrics: Dict[str, Any]
    threshold: float
    paths: Dict[str, str]

def nested_oof_predictions_for_config(X, y, pca_variant, selector_name, clf_name, model_id, feature_names, gparams):

    outer_cv = RepeatedStratifiedKFold(
        n_splits=OUTER_FOLDS,
        n_repeats=OUTER_REPEATS,
        random_state=RANDOM_SEED
    )

    inner_cv = RepeatedStratifiedKFold(
        n_splits=INNER_FOLDS,
        n_repeats=INNER_REPEATS,
        random_state=RANDOM_SEED
    )

    oof_score = np.zeros(len(y), dtype=float)
    oof_thr   = np.zeros(len(y), dtype=float)

    fold_info = []
    selected_sets = []

    fold_val_aucs = []

    # --- NEW: per-fold sanity aggregation collectors ---
    fold_sanity_counts = []      # var/aucf/corr/fsu/pca per fold
    fold_fsu_counts = []         # selector list sizes + consensus kept per fold

    # NEW: detailed fold metrics (TRAIN/VAL/TEST)
    fold_metrics_rows = []

    # ============================================================
    # 2) NEW: Collect per-fold selector outputs (inside fsu)
    # ============================================================
    # Long-form rows: one row per (fold, selector_method, feature)
    fold_selector_feature_rows = []
    # Long-form rows: one row per (fold, consensus_feature) with vote count
    fold_consensus_vote_rows = []
    fold_shap_rows = []

    # NEW: track per-fold PCA_n and selector_k for stability summary
    fold_pca_n = []

    # ---- helper: feature names right BEFORE a given step (e.g., "fsu") ----
    def _feature_names_before_step(fitted_pipe, original_feature_names, stop_step: str):
        names = list(original_feature_names)

        if not hasattr(fitted_pipe, "steps"):
            return names

        for name, step in fitted_pipe.steps:
            lname = name.lower()

            # stop BEFORE this step
            if lname == str(stop_step).lower():
                break

            if step is None or (isinstance(step, str) and step == "passthrough"):
                continue

            # skip steps that do not change feature set (or we don't want to track here)
            if lname in ("sample", "sampler", "balance", "scaler", "normalize", "norm", "scale", "impute", "clf"):
                continue

            # PCA changes names to PCs; if PCA happens before stop_step, reflect it
            if "pca" in lname:
                n_comp = getattr(step, "n_components_", None)
                if n_comp is None:
                    n_comp = getattr(step, "n_components", None)
                if n_comp is None:
                    continue
                try:
                    n_comp_int = int(n_comp)
                except Exception:
                    n_comp_int = int(getattr(step, "n_components_", len(names)))
                names = [f"PC{i+1}" for i in range(n_comp_int)]
                continue

            idx = _step_support_indices(step, len(names))
            idx = np.asarray(idx, dtype=int)
            idx = idx[(idx >= 0) & (idx < len(names))]
            names = [names[i] for i in idx]

        return names

    for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"  Outer fold {fold}/{OUTER_FOLDS}...")

        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[va_idx], y[va_idx]

        # selector_name is just a label; the actual combo comes from 'selector_name' parsing OR passed directly.
        # We pass combo directly from outside by adding a new argument if you want; simplest: parse selector_name here:
        fs_methods = tuple(selector_name.replace("FSU__", "").split("+")) if selector_name.startswith("FSU__") else tuple(FS_METHODS_ALL)

        pipe, dist = build_pipeline(pca_variant, selector_name, clf_name, fs_methods=fs_methods)
        pipe.set_params(**gparams)

        # fold-adaptive SMOTE (safe for small minority folds)
        if (BALANCING_METHOD or "").lower().strip() == "smote":
            pipe.set_params(sample=make_sampler(y_for_fold=y_tr))

        search = make_search(pipe, dist, inner_cv)

        if search is None:
            best_est = pipe.fit(X_tr, y_tr)
            best_params = {}
        else:
            search.fit(X_tr, y_tr)
            best_est = search.best_estimator_
            best_params = search.best_params_

        # always track PCA_n (even when no search)
        fold_pca_n.append(pca_n_components_from_est(best_est))



        selected_names = extract_selected_feature_names(best_est, feature_names)
        selected_sets.append(set(selected_names))

        # NEW: SHAP on TRAIN fold only (leak-safe), classifier-input space
        # (fast path: trees + linear only)
        sh_rows = compute_fold_mean_abs_shap(best_est, X_tr, y_tr, max_samples=200, perm_repeats=5)
        print(f"  [SHAP] fold {fold}: clf={clf_name} | n_rows={len(sh_rows)}")


        # Keep only selected consensus features (safe only when PCA is not active)
        pca_step = best_est.named_steps.get("pca", None)
        pca_active = (pca_step is not None) and not (isinstance(pca_step, str) and pca_step == "passthrough")

        if (not pca_active) and selected_names:
            sel_set = set(map(str, selected_names))
            sh_rows = [r for r in sh_rows if str(r.get("feature", "")) in sel_set]

        for r in sh_rows:
            r.update({
                "model_id": model_id,
                "pca": pca_variant,
                "selector": selector_name,
                "classifier": clf_name,
                "fold": int(fold),
            })
        fold_shap_rows.extend(sh_rows)



        # --- SANITY: number of features after each preprocessing step ---
        cnt = feature_counts_by_step(best_est, X_tr, steps=("var","aucf","corr","fsu","pca"))
        print(
            "  [SANITY] n_features after: "
            f"var={cnt['var']} | "
            f"aucf={cnt['aucf']} | "
            f"corr={cnt['corr']} | "
            f"fsu={cnt['fsu']} | "
            f"pca={cnt['pca']}"
        )


        fold_sanity_counts.append({
            "model_id": model_id,
            "pca_variant": pca_variant,
            "selector": selector_name,
            "classifier": clf_name,
            "fold": int(fold),
            "var": cnt.get("var", None),
            "aucf": cnt.get("aucf", None),
            "corr": cnt.get("corr", None),
            "fsu": cnt.get("fsu", None),
            "pca_n": cnt.get("pca", None),
        })

        # ============================================================
        # SANITY-ID: are corr and fsu selecting the exact same features?
        # (needs helper feature_names_before_step defined globally)
        # ============================================================
        # features entering FSU
        names_before_fsu = feature_names_before_step(best_est, feature_names, stop_step="fsu")

        # features leaving FSU (i.e., entering sampler)
        names_after_fsu = feature_names_before_step(best_est, feature_names, stop_step="sample")

        print("  [SANITY-ID] fsu changed features:", set(names_before_fsu) != set(names_after_fsu))
        print("  [SANITY] features before FSU:", len(names_before_fsu),
              " after FSU:", len(names_after_fsu))

        print("  [SANITY-ID] fsu-dropped features (first 20):",
              sorted(set(names_before_fsu) - set(names_after_fsu))[:20])

        # ============================================================
        # NEW: per-fold selector outputs (fsu.details_) + consensus votes
        # ============================================================
        fsu_step = None
        try:
            fsu_step = best_est.named_steps.get("fsu", None)
        except Exception:
            fsu_step = None

        if fsu_step is not None and hasattr(fsu_step, "details_") and isinstance(fsu_step.details_, dict):
            det = fsu_step.details_
            names_before_fsu = _feature_names_before_step(best_est, feature_names, stop_step="fsu")

            selector_keys = ["N_MRMR", "mRMR", "N_BORUTA", "N_L1", "N_ENET", "RFE"]
            for ksel in selector_keys:
                idx_list = det.get(ksel, [])
                if not isinstance(idx_list, (list, tuple, np.ndarray)):
                    idx_list = []
                selector_method = "RFE" if ksel == "RFE" else ksel

                for ii in idx_list:
                    try:
                        ii = int(ii)
                    except Exception:
                        continue
                    if 0 <= ii < len(names_before_fsu):
                        fold_selector_feature_rows.append({
                            "model_id": model_id,
                            "pca": pca_variant,
                            "selector": selector_name,
                            "classifier": clf_name,
                            "fold": int(fold),
                            "selector_method": str(selector_method),
                            "feature": str(names_before_fsu[ii]),
                            "rank_within_selector": None,
                        })

            votes_dict = det.get("CONSENSUS_VOTES", {})
            if isinstance(votes_dict, dict):
                for ii, vv in votes_dict.items():
                    try:
                        ii = int(ii); vv = int(vv)
                    except Exception:
                        continue
                    if 0 <= ii < len(names_before_fsu):
                        fold_consensus_vote_rows.append({
                            "model_id": model_id,
                            "pca": pca_variant,
                            "selector": selector_name,
                            "classifier": clf_name,
                            "fold": int(fold),
                            "feature": str(names_before_fsu[ii]),
                            "votes": int(vv),
                        })

            # ✅ append ONCE per fold (outside loops)
            fold_fsu_counts.append({
                "model_id": model_id,
                "pca": pca_variant,
                "selector": selector_name,
                "classifier": clf_name,
                "fold": int(fold),
                "N_MRMR": int(len(det.get("N_MRMR", []) or [])),
                "mRMR": int(len(det.get("mRMR", []) or [])),
                "N_BORUTA": int(len(det.get("N_BORUTA", []) or [])),
                "N_L1": int(len(det.get("N_L1", []) or [])),
                "N_ENET": int(len(det.get("N_ENET", []) or [])),
                "RFE": int(len(det.get("RFE", []) or [])),
                "CONSENSUS_KEPT": int(len(selected_names)),
            })
        # --- NEW: per-fold consensus feature list (count + names) ---
        print(f"  [FOLD {fold}] Consensus features kept: {len(selected_names)}")
        if len(selected_names) > 0:
            print("  [FOLD {}] Feature names: {}".format(fold, ", ".join(map(str, selected_names))))

            # --- NEW: print per-feature univariate AUC on TRAIN fold for the selected consensus features ---
            if any(str(n).startswith("PC") for n in selected_names):
                print(f"  [FOLD {fold}] PCA active → skipping per-feature univariate AUC print.")
            else:
                print(f"  [FOLD {fold}] Univariate AUC per selected feature (TRAIN fold):")
                for fn in selected_names:
                    try:
                        col = pd.to_numeric(X_tr[fn], errors="coerce").values.astype(float)
                        col2 = col.copy()
                        col2[~np.isfinite(col2)] = np.nan
                        med = np.nanmedian(col2)
                        if not np.isfinite(med):
                            med = 0.0
                        col2 = np.where(np.isfinite(col2), col2, med)

                        if np.nanstd(col2) < 1e-12 or len(np.unique(y_tr)) != 2:
                            aucv = np.nan
                        else:
                            aucv = roc_auc_score(y_tr, col2)

                        print(f"    {fn}: {aucv:.4f}" if np.isfinite(aucv) else f"    {fn}: nan")
                    except Exception:
                        print(f"    {fn}: nan")

        # ---- scores ----
        tr_score = predict_score(best_est, X_tr)
        thr = optimal_threshold_youden(y_tr, tr_score)

        va_score = predict_score(best_est, X_va)

        oof_score[va_idx] = va_score
        oof_thr[va_idx] = thr

        # fold validation AUC (for mean/std)
        try:
            fold_auc = roc_auc_score(y_va, va_score) if len(np.unique(y_va)) == 2 else np.nan
        except Exception:
            fold_auc = np.nan
        fold_val_aucs.append(float(fold_auc))

        fold_info.append({"fold": fold, "best_params": best_params, "thr": thr, "val_auc": float(fold_auc)})

        # ---- per-fold metrics at the SAME threshold (thr) ----
        tr_m = compute_metrics(y_tr, tr_score, threshold=thr)
        va_m = compute_metrics(y_va, va_score, threshold=thr)

        def _pick(m, prefix):
            return {f"{prefix}_{k}": v for k, v in m.items()}

        # per-fold dicts for saving into fold_metrics.csv
        sanity_counts_fold = fold_sanity_counts[-1] if len(fold_sanity_counts) else {}
        fsu_counts_fold = fold_fsu_counts[-1] if len(fold_fsu_counts) else {}

        row = {
            "model_id": model_id,
            "pca": pca_variant,
            "selector": selector_name,
            "classifier": clf_name,
            "fold": fold,
            "fold_thr_from_train": float(thr),
            "fold_val_auc": float(fold_auc),

            # store *this fold* counts (json)
            "sanity_counts_json": json.dumps([sanity_counts_fold], default=str),
            "fsu_counts_json": json.dumps([fsu_counts_fold], default=str),
        }

        row.update(_pick(tr_m, "train"))
        row.update(_pick(va_m, "val"))
        fold_metrics_rows.append(row)


    # NEW: stability summary for PCA_n across outer folds
    pca_arr = np.asarray([np.nan if v is None else float(v) for v in fold_pca_n], dtype=float)
    pca_mean = float(np.nanmean(pca_arr)) if np.isfinite(pca_arr).any() else float("nan")
    pca_std  = float(np.nanstd(pca_arr, ddof=1)) if np.sum(np.isfinite(pca_arr)) >= 2 else float("nan")
    print(f"  [STABILITY] {model_id} | PCA_n: {pca_mean:.2f}±{pca_std:.2f}")

    # --- OOF validation metrics (overall, fold-specific thresholds) ---
    val_auc = roc_auc_score(y, oof_score) if len(np.unique(y)) == 2 else float("nan")
    y_pred = (oof_score >= oof_thr).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    youden = sens + spec - 1.0

    fold_val_aucs_arr = np.asarray(fold_val_aucs, dtype=float)
    val_auc_mean = float(np.nanmean(fold_val_aucs_arr))
    val_auc_std  = float(np.nanstd(fold_val_aucs_arr, ddof=1)) if np.sum(~np.isnan(fold_val_aucs_arr)) >= 2 else float("nan")

    val_metrics = {
        "AUC_OOF": float(val_auc),
        "AUC_FOLD_MEAN": float(val_auc_mean),
        "AUC_FOLD_STD": float(val_auc_std),

        "PCA_N_MEAN": float(pca_mean),
        "PCA_N_STD": float(pca_std),

        "ACC_OOF": float(accuracy_score(y, y_pred)),
        "BACC_OOF": float(balanced_accuracy_score(y, y_pred)),
        "MCC_OOF": float(matthews_corrcoef(y, y_pred)),
        "YOUDEN_OOF": float(youden),

        "THRESH_NOTE": "fold-specific thresholds from outer-train",
        "TN_OOF": int(tn),
        "FP_OOF": int(fp),
        "FN_OOF": int(fn),
        "TP_OOF": int(tp),
    }

    stability = {
        "selected_features_per_fold": selected_sets,
        "stability_jaccard_mean": jaccard_mean(selected_sets),
        "stability_top": Counter([f for s in selected_sets for f in s]).most_common(30)
    }

    # NEW: averages across folds for TRAIN/VAL metrics (at fold-threshold)
    fold_df = pd.DataFrame(fold_metrics_rows)
    avg_cols = [c for c in fold_df.columns if c.startswith("train_") or c.startswith("val_")]
    fold_metrics_avg = fold_df[avg_cols].mean(numeric_only=True).to_dict()

    # Return selector outputs too (so you can save CSVs per model outside)
    return (
        oof_score,
        oof_thr,
        val_metrics,
        fold_info,
        stability,
        fold_metrics_rows,
        fold_metrics_avg,
        fold_selector_feature_rows,   # NEW
        fold_consensus_vote_rows,
        fold_shap_rows,
        fold_sanity_counts,   # NEW
        fold_fsu_counts,      # NEW
    )


def fit_final_and_test_for_config(X_train, y_train, X_test, y_test, pca_variant, selector_name, clf_name, gparams):
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED)

    fs_methods = tuple(selector_name.replace("FSU__", "").split("+")) if selector_name.startswith("FSU__") else tuple(FS_METHODS_ALL)
    pipe, dist = build_pipeline(pca_variant, selector_name, clf_name, fs_methods=fs_methods)
    pipe.set_params(**gparams)

    # If you want dynamic SMOTE k, you must base it on y_train here (no fold y_tr exists)
    if (BALANCING_METHOD or "").lower().strip() == "smote":
        vals, counts = np.unique(np.asarray(y_train).astype(int), return_counts=True)
        if len(counts) == 2:
            minority = int(np.min(counts))
            k = max(1, min(5, minority - 1))
        else:
            k = 1
        pipe.set_params(sample=SMOTE(random_state=RANDOM_SEED, k_neighbors=k))

    search = make_search(pipe, dist, inner_cv)

    if search is None:
        best_est = pipe.fit(X_train, y_train)
        best_params = {}
    else:
        search.fit(X_train, y_train)
        best_est = search.best_estimator_
        best_params = search.best_params_

    train_score = predict_score(best_est, X_train)
    test_score  = predict_score(best_est, X_test)

    # In-sample DEV threshold (keep for TRAIN reporting; do NOT use for unbiased TEST thresholding)
    thr_train = optimal_threshold_youden(y_train, train_score)

    train_metrics = compute_metrics(y_train, train_score, threshold=thr_train)

    # Keep your historical behavior for test_metrics (DEV train threshold)
    test_metrics  = compute_metrics(y_test, test_score, threshold=thr_train)

    return best_est, best_params, thr_train, train_score, test_score, train_metrics, test_metrics



def _fmt_metrics(m: dict) -> str:
    # Safe formatting even if something is missing/NaN
    def g(k, default=np.nan):
        v = m.get(k, default)
        try:
            return float(v)
        except Exception:
            return np.nan

    return (
        f"AUC={g('AUC'):.3f} "
        f"BACC={g('BACC'):.3f} "
        f"MCC={g('MCC'):.3f} "
        f"YOUDEN={g('YOUDEN'):.3f}"
    )

def _fmt_line(m: dict) -> str:
    # prints all keys you currently compute in compute_metrics()
    keys = ["AUC","ACC","BACC","MCC","SENS","SPEC","YOUDEN","PREC","RECALL","F1","THRESH","TN","FP","FN","TP"]
    parts = []
    for k in keys:
        v = m.get(k, None)
        if v is None:
            continue
        if isinstance(v, float):
            parts.append(f"{k}={v:.3f}")
        else:
            parts.append(f"{k}={v}")
    return " | ".join(parts)

def show_fold_report(fold_rows, model_id: str, max_rows=50):
    df = pd.DataFrame(fold_rows).copy()
    if df.empty:
        print(f"[{model_id}] No fold rows to display.")
        return

    # keep fold ordering
    df = df.sort_values("fold").reset_index(drop=True)

    # compact columns for viewing
    view_cols = [
        "fold", "fold_thr_from_train", "fold_val_auc",
        "train_AUC","train_ACC","train_BACC","train_MCC","train_SENS","train_SPEC","train_YOUDEN",
        "val_AUC","val_ACC","val_BACC","val_MCC","val_SENS","val_SPEC","val_YOUDEN",
        "train_TN","train_FP","train_FN","train_TP",
        "val_TN","val_FP","val_FN","val_TP",
    ]
    view_cols = [c for c in view_cols if c in df.columns]

    print(f"\n[{model_id}] === PER-FOLD METRICS (threshold = fold_thr_from_train) ===")
    display(df[view_cols].head(max_rows))

    # fold means/stds (very useful)
    num_cols = [c for c in df.columns if c.startswith("train_") or c.startswith("val_") or c in ("fold_val_auc",)]
    agg = df[num_cols].agg(["mean", "std"]).T.reset_index().rename(columns={"index":"metric"})
    print(f"\n[{model_id}] === FOLD SUMMARY (mean±std) ===")
    display(agg)

def show_model_report(model_id: str,
                      val_metrics: dict,
                      val_metrics_at_thr: dict,
                      train_metrics: dict,
                      test_metrics: dict,
                      thr_oof: float):
    print(f"\n[{model_id}] === VALIDATION (OOF, fold-specific thresholds) ===")
    # your val_metrics dict uses *_OOF naming, print them clearly
    for k in ["AUC_OOF","ACC_OOF","BACC_OOF","MCC_OOF","YOUDEN_OOF","TN_OOF","FP_OOF","FN_OOF","TP_OOF",
              "AUC_FOLD_MEAN","AUC_FOLD_STD","PCA_N_MEAN","PCA_N_STD"]:
        if k in val_metrics:
            v = val_metrics[k]
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    print(f"\n[{model_id}] === VALIDATION (OOF, ONE GLOBAL THRESHOLD from OOF) ===")
    print(f"  thr_oof={thr_oof:.6f}")
    print("  " + _fmt_line(val_metrics_at_thr))

    print(f"\n[{model_id}] === TRAIN (final refit on full DEV, thr from DEV train) ===")
    print("  " + _fmt_line(train_metrics))

    print(f"\n[{model_id}] === TEST (holdout, evaluated at DEV train thr) ===")
    print("  " + _fmt_line(test_metrics))

def make_fsu_union_summary_csv(sel_rows, vote_rows, out_csv_path, shap_rows=None):
    """
    Build a compact "union summary" across outer folds.
    Inputs:
      - sel_rows: list of dicts (fold, selector_method, feature)  [your FSU_selector_outputs_long]
      - vote_rows: list of dicts (fold, feature, votes)           [your FSU_consensus_votes_long]
      - shap_rows: optional list of dicts (fold, feature, mean_abs_shap)
    Output columns:
      feature
      n_folds_selected_any_selector
      n_folds_in_consensus
      mean_selector_votes_in_consensus
      max_selector_votes_in_consensus
      n_selectors_voted
      selectors_voted
      selectors_by_fold
      mean_abs_shap   (optional)
    """
    df_sel = pd.DataFrame(sel_rows) if sel_rows else pd.DataFrame(columns=["fold","selector_method","feature"])
    df_vot = pd.DataFrame(vote_rows) if vote_rows else pd.DataFrame(columns=["fold","feature","votes"])

    if df_sel.empty and df_vot.empty:
        # still write an empty file (nice for pipelines)
        pd.DataFrame(columns=[
            "feature",
            "n_folds_selected_any_selector",
            "n_folds_in_consensus",
            "mean_selector_votes_in_consensus",
            "max_selector_votes_in_consensus",
            "n_selectors_voted",
            "selectors_voted",
            "selectors_by_fold",
            "mean_abs_shap",
        ]).to_csv(out_csv_path, index=False)
        return

    # ---- Any-selector presence per fold ----
    if not df_sel.empty:
        df_sel["fold"] = df_sel["fold"].astype(int)
        df_sel["feature"] = df_sel["feature"].astype(str)
        df_sel["selector_method"] = df_sel["selector_method"].astype(str)

        folds_any = (
            df_sel.groupby("feature")["fold"]
            .nunique()
            .rename("n_folds_selected_any_selector")
            .reset_index()
        )

        # union of selectors across folds
        selectors_union = (
            df_sel.groupby("feature")["selector_method"]
            .apply(lambda s: "|".join(sorted(set(s))))
            .rename("selectors_voted")
            .reset_index()
        )
        n_selectors_union = (
            df_sel.groupby("feature")["selector_method"]
            .nunique()
            .rename("n_selectors_voted")
            .reset_index()
        )

        # per-fold selector breakdown: F1:N_MRMR|RFE;F2:mRMR|N_L1 ...
        by_fold = (
            df_sel.groupby(["feature","fold"])["selector_method"]
            .apply(lambda s: "|".join(sorted(set(s))))
            .reset_index()
        )
        selectors_by_fold = (
            by_fold.groupby("feature")
            .apply(lambda d: ";".join([f"F{int(r.fold)}:{r.selector_method}" for r in d.itertuples(index=False)]))
            .rename("selectors_by_fold")
            .reset_index()
        )
    else:
        folds_any = pd.DataFrame(columns=["feature","n_folds_selected_any_selector"])
        selectors_union = pd.DataFrame(columns=["feature","selectors_voted"])
        n_selectors_union = pd.DataFrame(columns=["feature","n_selectors_voted"])
        selectors_by_fold = pd.DataFrame(columns=["feature","selectors_by_fold"])

    # ---- Consensus presence + vote stats (votes are PER FOLD, as you assumed) ----
    if not df_vot.empty:
        df_vot["fold"] = df_vot["fold"].astype(int)
        df_vot["feature"] = df_vot["feature"].astype(str)
        df_vot["votes"] = df_vot["votes"].astype(int)

        folds_cons = (
            df_vot.groupby("feature")["fold"]
            .nunique()
            .rename("n_folds_in_consensus")
            .reset_index()
        )

        vote_mean = (
            df_vot.groupby("feature")["votes"]
            .mean()
            .rename("mean_selector_votes_in_consensus")
            .reset_index()
        )
        vote_max = (
            df_vot.groupby("feature")["votes"]
            .max()
            .rename("max_selector_votes_in_consensus")
            .reset_index()
        )
    else:
        folds_cons = pd.DataFrame(columns=["feature","n_folds_in_consensus"])
        vote_mean = pd.DataFrame(columns=["feature","mean_selector_votes_in_consensus"])
        vote_max  = pd.DataFrame(columns=["feature","max_selector_votes_in_consensus"])

    # ---- Merge all ----
    # Start from all features seen anywhere
    all_feats = pd.Series(
        pd.concat([
            df_sel["feature"] if "feature" in df_sel.columns else pd.Series(dtype=str),
            df_vot["feature"] if "feature" in df_vot.columns else pd.Series(dtype=str),
        ], ignore_index=True).unique(),
        name="feature"
    ).to_frame()

    out = (all_feats
           .merge(folds_any, on="feature", how="left")
           .merge(folds_cons, on="feature", how="left")
           .merge(vote_mean, on="feature", how="left")
           .merge(vote_max,  on="feature", how="left")
           .merge(n_selectors_union, on="feature", how="left")
           .merge(selectors_union,  on="feature", how="left")
           .merge(selectors_by_fold, on="feature", how="left")
          )

    # Fill defaults
    out["n_folds_selected_any_selector"] = out["n_folds_selected_any_selector"].fillna(0).astype(int)
    out["n_folds_in_consensus"] = out["n_folds_in_consensus"].fillna(0).astype(int)
    out["mean_selector_votes_in_consensus"] = out["mean_selector_votes_in_consensus"].fillna(np.nan)
    out["max_selector_votes_in_consensus"]  = out["max_selector_votes_in_consensus"].fillna(np.nan)
    out["n_selectors_voted"] = out["n_selectors_voted"].fillna(0).astype(int)
    out["selectors_voted"] = out["selectors_voted"].fillna("")
    out["selectors_by_fold"] = out["selectors_by_fold"].fillna("")


    # Optional SHAP column (ALWAYS create column if shap_rows is provided)
    if shap_rows is not None:
        df_sh = pd.DataFrame(shap_rows)
        if not df_sh.empty:
            df_sh["feature"] = df_sh["feature"].astype(str)
            sh_agg = (df_sh.groupby("feature")["mean_abs_shap"]
                      .mean()
                      .rename("mean_abs_shap")
                      .reset_index())
            out = out.merge(sh_agg, on="feature", how="left")
        else:
            out["mean_abs_shap"] = np.nan

    # Sort: most stable in consensus first, then by selector breadth, then by any-selector fold count
    sort_cols = ["n_folds_in_consensus", "max_selector_votes_in_consensus",
                 "n_selectors_voted", "n_folds_selected_any_selector", "feature"]
    asc = [False, False, False, False, True]
    out = out.sort_values(sort_cols, ascending=asc).reset_index(drop=True)

    out.to_csv(out_csv_path, index=False)


    # ============================================================
# FREEZE / SAVE FINAL SINGLE-CLASSIFIER MODELS
# ============================================================
TARGET_FROZEN_CLASSIFIERS = [
    "NAIVE_BAYES",
    "GAUSSIAN_PROCESS",
    "EXTRATREES",
    "LOGREG",
    "SGD_LOGLOSS",
]

def save_frozen_single_model(
    save_path,
    fitted_pipe,
    classifier_name,
    model_id,
    feature_columns,
    threshold,
    best_params,
    global_params,
    val_auc_mean=None,
    val_auc_std=None,
    val_mcc=None,
):
    pkg = {
        "classifier_name": str(classifier_name),
        "model_id": str(model_id),
        "pipeline": fitted_pipe,                    # FULL fitted pipeline
        "feature_columns": list(feature_columns),   # raw input columns expected at inference
        "threshold": float(threshold),
        "best_params": best_params if isinstance(best_params, dict) else {},
        "global_params": global_params if isinstance(global_params, dict) else {},
        "val_auc_mean": None if val_auc_mean is None else float(val_auc_mean),
        "val_auc_std": None if val_auc_std is None else float(val_auc_std),
        "val_mcc": None if val_mcc is None else float(val_mcc),
    }
    joblib.dump(pkg, save_path)
    print(f"[FROZEN MODEL] Saved {classifier_name} -> {save_path}")


def coerce_best_params_field(x):
    if isinstance(x, dict):
        return {k: _coerce_param_value(v) for k, v in x.items()}
    if isinstance(x, str):
        try:
            d = json.loads(x)
            if isinstance(d, dict):
                return {k: _coerce_param_value(v) for k, v in d.items()}
        except Exception:
            pass
    return {}


def refit_and_freeze_best_model_per_classifier(
    results_df,
    X_train,
    y_train,
    feature_names,
    out_dir,
    target_classifiers,
):
    frozen_dir = os.path.join(out_dir, "FROZEN_SINGLE_MODELS")
    os.makedirs(frozen_dir, exist_ok=True)

    frozen_summary_rows = []

    for clf_name in target_classifiers:
        sub = results_df[results_df["classifier"] == clf_name].copy()

        if sub.empty:
            print(f"[FROZEN MODEL] No rows found for classifier: {clf_name}")
            continue

        # choose best config by validation only
        sub = sub.sort_values(
            ["val_AUC_mean", "val_AUC_std", "val_MCC", "val_oof_AUC"],
            ascending=[False, True, False, False]
        ).reset_index(drop=True)

        best_row = sub.iloc[0]

        model_id = best_row["model_id"]
        pca_variant = best_row.get("pca", "no_pca")
        selector_name = best_row.get("selector", "NONE")

        best_params = coerce_best_params_field(best_row.get("best_params", {}))
        global_params = coerce_best_params_field(best_row.get("global_params", {}))

        fs_methods = (
            tuple(selector_name.replace("FSU__", "").split("+"))
            if str(selector_name).startswith("FSU__")
            else tuple(FS_METHODS_ALL)
        )

        pipe, _ = build_pipeline(
            pca_variant=pca_variant,
            selector_name=selector_name,
            clf_name=clf_name,
            fs_methods=fs_methods,
        )

        if global_params:
            pipe.set_params(**global_params)
        if best_params:
            pipe.set_params(**best_params)

        # safe dynamic SMOTE if ever used
        if (BALANCING_METHOD or "").lower().strip() == "smote":
            vals, counts = np.unique(np.asarray(y_train).astype(int), return_counts=True)
            if len(counts) == 2:
                minority = int(np.min(counts))
                k = max(1, min(5, minority - 1))
            else:
                k = 1
            pipe.set_params(sample=SMOTE(random_state=RANDOM_SEED, k_neighbors=k))

        fitted_pipe = pipe.fit(X_train, y_train)

        train_score = predict_score(fitted_pipe, X_train)
        thr_train = optimal_threshold_youden(y_train, train_score)

        save_path = os.path.join(frozen_dir, f"{clf_name}__frozen_model.joblib")
        save_frozen_single_model(
            save_path=save_path,
            fitted_pipe=fitted_pipe,
            classifier_name=clf_name,
            model_id=model_id,
            feature_columns=feature_names,
            threshold=thr_train,
            best_params=best_params,
            global_params=global_params,
            val_auc_mean=best_row.get("val_AUC_mean", np.nan),
            val_auc_std=best_row.get("val_AUC_std", np.nan),
            val_mcc=best_row.get("val_MCC", np.nan),
        )

        frozen_summary_rows.append({
            "classifier": clf_name,
            "model_id": model_id,
            "save_path": save_path,
            "threshold": float(thr_train),
            "val_AUC_mean": best_row.get("val_AUC_mean", np.nan),
            "val_AUC_std": best_row.get("val_AUC_std", np.nan),
            "val_MCC": best_row.get("val_MCC", np.nan),
            "best_params": json.dumps(best_params, default=str),
            "global_params": json.dumps(global_params, default=str),
        })

    frozen_summary_df = pd.DataFrame(frozen_summary_rows)
    frozen_summary_path = os.path.join(frozen_dir, "frozen_models_summary.csv")
    frozen_summary_df.to_csv(frozen_summary_path, index=False)
    print(f"[FROZEN MODEL] Summary saved: {frozen_summary_path}")

    return frozen_summary_df, frozen_dir

# ============================================================
# 9) RUN ALL CONFIGS
# ============================================================
def n_features_before_clf(fitted_pipe, X_sample_df):
    """
    Count how many features survive preprocessing + selection,
    right before sampler / classifier.
    """
    Xcur = X_sample_df
    for name, step in fitted_pipe.steps:
        if name.lower() in ("sample", "sampler", "clf"):
            break
        if step is None or (isinstance(step, str) and step == "passthrough"):
            continue
        Xcur = step.transform(Xcur)
    return int(np.asarray(Xcur).shape[1])


Xtr = X_train.copy()
Xte = X_test.copy()
summary_rows = []

TOTAL_MODELS = len(CLASSIFIERS) * len(PCA_VARIANTS) * len(GLOBAL_COMBOS) * (len(FS_COMBOS) if USE_UNION_FS else 1)
model_counter = 0

for pca_variant in PCA_VARIANTS:
    for clf_name in CLASSIFIERS:
        for gi, gparams in enumerate(GLOBAL_COMBOS, start=1):
            # if FSU off, run a single passthrough config
            combos = FS_COMBOS if USE_UNION_FS else [()]

            for ci, combo in enumerate(combos, start=1):
                selector_name = "FSU__" + "+".join(combo) if combo else "NONE"
                model_id = f"{pca_variant}__{clf_name}__G{gi:02d}__S{ci:02d}"
                model_counter += 1
                print(f"\n=== Running ({model_counter}/{TOTAL_MODELS}): {model_id} ===")
                print("GLOBAL:", gparams)
                print("FS_COMBO:", combo)

                oof_score, oof_thr, val_metrics, fold_info, stability, fold_rows, fold_avg, sel_rows, vote_rows, shap_rows, sanity_counts, fsu_counts = (
                    nested_oof_predictions_for_config(
                        Xtr, y_train, pca_variant, selector_name, clf_name,
                        model_id, feature_names, gparams,
                    )
                )

                best_est, best_params, thr, train_score, test_score, train_metrics, test_metrics = (
                    fit_final_and_test_for_config(
                        Xtr, y_train, Xte, y_test, pca_variant, selector_name, clf_name, gparams
                    )
                )

                print(f"[{model_id}] PCA_n={pca_n_components_from_est(best_est)}")

                print(
                    f"[{model_id}] AUCs: "
                    f"TRAIN={train_metrics['AUC']:.3f} | "
                    f"TEST={test_metrics['AUC']:.3f}"
                )

                thr_oof = optimal_threshold_youden(y_train, oof_score)
                val_metrics_at_thr = compute_metrics(y_train, oof_score, threshold=thr_oof)

                # TEST evaluated at the DEV-OOF-derived threshold (preferred reporting)
                thr_dev_oof = thr_oof
                test_metrics_oofthr = compute_metrics(y_test, test_score, threshold=thr_dev_oof)


                # ---- NEW: rich notebook reporting ----
                show_fold_report(fold_rows, model_id=model_id)

                show_model_report(
                    model_id=model_id,
                    val_metrics=val_metrics,
                    val_metrics_at_thr=val_metrics_at_thr,
                    train_metrics=train_metrics,
                    test_metrics=test_metrics,
                    thr_oof=thr_oof,
                )

                print(
                    f"[{model_id}] TEST@OOFthr(thr={thr_dev_oof:.4f}) "
                    f"AUC={test_metrics_oofthr['AUC']:.3f} "
                    f"BACC={test_metrics_oofthr['BACC']:.3f} "
                    f"MCC={test_metrics_oofthr['MCC']:.3f} "
                    f"YOUDEN={test_metrics_oofthr['YOUDEN']:.3f} "
                    f"ACC={test_metrics_oofthr['ACC']:.3f}"
                )

                print(
                    f"[{model_id}] VAL@OOFthr(thr={thr_oof:.4f}) "
                    f"AUC={val_metrics_at_thr['AUC']:.3f} "
                    f"BACC={val_metrics_at_thr['BACC']:.3f} "
                    f"MCC={val_metrics_at_thr['MCC']:.3f} "
                    f"YOUDEN={val_metrics_at_thr['YOUDEN']:.3f}"
                )
                print(
                    f"[{model_id}] TEST@OOFthr(thr={thr_dev_oof:.4f}) "
                    f"AUC={test_metrics_oofthr['AUC']:.3f} "
                    f"ACC={test_metrics_oofthr['ACC']:.3f} "
                    f"BACC={test_metrics_oofthr['BACC']:.3f} "
                    f"MCC={test_metrics_oofthr['MCC']:.3f} "
                    f"YOUDEN={test_metrics_oofthr['YOUDEN']:.3f}"
                )

                # Save per-model outputs
                model_dir = os.path.join(OUT_DIR, model_id)
                os.makedirs(model_dir, exist_ok=True)

                # 1) Save the two detailed CSVs (as you already do)
                if sel_rows:
                    pd.DataFrame(sel_rows).to_csv(
                        os.path.join(model_dir, "FSU_selector_outputs_long.csv"),
                        index=False
                    )

                if vote_rows:
                    pd.DataFrame(vote_rows).to_csv(
                        os.path.join(model_dir, "FSU_consensus_votes_long.csv"),
                        index=False
                    )

                # 2) ALWAYS write the 3rd CSV if there is *any* FSU data
                # Save SHAP per model (optional)
                if shap_rows and len(shap_rows) > 0:
                    pd.DataFrame(shap_rows).to_csv(
                        os.path.join(model_dir, "FSU_shap_meanabs_long.csv"),
                        index=False
                    )

                union_summary_path = os.path.join(model_dir, "FSU_union_features_summary.csv")
                make_fsu_union_summary_csv(
                    sel_rows=sel_rows,
                    vote_rows=vote_rows,
                    out_csv_path=union_summary_path,
                    shap_rows=shap_rows,
                )
                print("Saved:", union_summary_path)

                # 3) Continue with your existing outputs (these should NOT be inside vote_rows)
                pd.DataFrame(fold_rows).to_csv(os.path.join(model_dir, "fold_metrics.csv"), index=False)
                pd.DataFrame([{"model_id": model_id, **fold_avg}]).to_csv(
                    os.path.join(model_dir, "fold_metrics_avg.csv"), index=False
                )

                pd.DataFrame({"patient_id": train_ids, "label": y_train, "score": train_score}).to_csv(
                    os.path.join(model_dir, "train_predictions.csv"), index=False
                )
                pd.DataFrame(
                    {"patient_id": train_ids, "label": y_train, "score": oof_score, "fold_threshold": oof_thr}
                ).to_csv(os.path.join(model_dir, "val_oof_predictions.csv"), index=False)
                pd.DataFrame({"patient_id": test_ids, "label": y_test, "score": test_score}).to_csv(
                    os.path.join(model_dir, "test_predictions.csv"), index=False
                )

                with open(os.path.join(model_dir, "meta.json"), "w") as f:
                    json.dump(
                        {
                            "model_id": model_id,
                            "pca_variant": pca_variant,
                            "selector": selector_name,
                            "classifier": clf_name,
                            "best_params": best_params,
                            "outer_fold_info": fold_info,
                            "threshold_from_full_train": thr,
                            "feature_stability": {
                            "jaccard_mean": stability["stability_jaccard_mean"],
                            "top": stability["stability_top"],
                            },
                        },
                        f,
                        indent=2,
                    )

                row = {
                    "model_id": model_id,
                    "pca": pca_variant,
                    "selector": selector_name,
                    "classifier": clf_name,
                    "best_params": json.dumps(best_params, default=str),
                    "global_params": json.dumps(gparams, default=str),
                    "global_combo_index": int(gi),

                    "thr_train": train_metrics["THRESH"],

                    "train_full_AUC": train_metrics["AUC"],
                    "train_ACC": train_metrics["ACC"],
                    "train_BACC": train_metrics["BACC"],
                    "train_MCC": train_metrics["MCC"],
                    "train_YOUDEN": train_metrics["YOUDEN"],

                    "val_oof_AUC": val_metrics["AUC_OOF"],
                    "val_AUC_mean": val_metrics["AUC_FOLD_MEAN"],
                    "val_AUC_std": val_metrics["AUC_FOLD_STD"],

                    "pca_n_mean": val_metrics.get("PCA_N_MEAN", np.nan),
                    "pca_n_std":  val_metrics.get("PCA_N_STD",  np.nan),

                    "val_ACC": val_metrics["ACC_OOF"],
                    "val_BACC": val_metrics["BACC_OOF"],
                    "val_MCC": val_metrics["MCC_OOF"],
                    "val_YOUDEN": val_metrics["YOUDEN_OOF"],

                    "test_AUC": test_metrics["AUC"],
                    "test_ACC": test_metrics["ACC"],
                    "test_BACC": test_metrics["BACC"],
                    "test_MCC": test_metrics["MCC"],
                    "test_YOUDEN": test_metrics["YOUDEN"],

                    "test_TN": test_metrics["TN"],
                    "test_FP": test_metrics["FP"],
                    "test_FN": test_metrics["FN"],
                    "test_TP": test_metrics["TP"],

                    "fold_metrics_csv": os.path.join(model_dir, "fold_metrics.csv"),
                    "fold_metrics_avg_csv": os.path.join(model_dir, "fold_metrics_avg.csv"),
                    "stability_jaccard_mean": stability["stability_jaccard_mean"],
                    "paths_dir": model_dir,
                    "thr_dev_oof": float(thr_dev_oof),

                    "test_ACC_oofthr": test_metrics_oofthr["ACC"],
                    "test_BACC_oofthr": test_metrics_oofthr["BACC"],
                    "test_MCC_oofthr": test_metrics_oofthr["MCC"],
                    "test_YOUDEN_oofthr": test_metrics_oofthr["YOUDEN"],
                                    }
                summary_rows.append(row)


def _coerce_param_value(v):
    if isinstance(v, str):
        s = v.strip()
        if s.lower() == "true": return True
        if s.lower() == "false": return False
        if s.lower() in ("none", "null"): return None
        if s.isdigit() or (s.startswith("-") and s[1:].isdigit()):
            try: return int(s)
            except Exception: pass
        try: return float(s)
        except Exception: return v
    return v

# ============================================================
# AFTER ALL MODELS
# ============================================================
results_df = pd.DataFrame(summary_rows)

results_path = os.path.join(OUT_DIR, "all_models_summary_metrics.csv")
results_df.to_csv(results_path, index=False)
print("\nSaved model-level metrics summary:", results_path)

# ============================================================
# FREEZE BEST MODEL FOR EACH OF THE 5 TARGET CLASSIFIERS
# ============================================================
frozen_summary_df, frozen_models_dir = refit_and_freeze_best_model_per_classifier(
    results_df=results_df,
    X_train=X_train,
    y_train=y_train,
    feature_names=feature_names,
    out_dir=OUT_DIR,
    target_classifiers=TARGET_FROZEN_CLASSIFIERS,
)

print("\n[FROZEN MODEL] Final frozen single-classifier models:")
display(frozen_summary_df)

# ============================================================
# NEW: POST-HOC FEATURE IMPORTANCE → UNION TOP FEATURES → FINAL REFIT
# ============================================================

if not USE_POSTHOC_IMPORTANCE_FS:
    print("\n[POST-HOC FS] Skipped (USE_POSTHOC_IMPORTANCE_FS=False).")

else:
    per_model_feats = {}
    all_top_feats = []
    per_model_impdf = {}
    top_models = pd.DataFrame()

    IMPORTANCE_CAPABLE = {
        "LOGREG", "SGD_LOGLOSS", "LDA",
        "RANDOM_FOREST", "DECISION_TREE", "EXTRATREES",
        "ADABOOST", "HIST_GBDT",
        "XGBOOST", "LIGHTGBM", "CATBOOST",
    }

    cand = results_df.copy()
    cand = cand[cand["classifier"].isin(IMPORTANCE_CAPABLE)].copy()

    if cand.empty:
        print("\n[POST-HOC FS] No importance-capable models found in results_df. Skipping post-hoc stage.")

    else:
        # If you ever re-enable PCA variants and want original-feature importances only:
        # cand = cand[cand["pca"] == "no_pca"].copy()

        cand = cand.sort_values(["val_AUC_mean", "val_AUC_std", "val_MCC"],
                                ascending=[False, True, False])
        top_models = cand.head(TOP_IMPORTANCE_MODELS)

        print("\n" + "=" * 90)
        print("[POST-HOC FS] Top importance-capable models (by val_AUC_mean):")
        display(top_models[["model_id", "classifier", "val_AUC_mean", "val_AUC_std", "val_MCC"]])

        TOPK_PRINT = 50

        # ----------------------------
        # 1) Refit each chosen model on FULL DEV and extract top features
        # ----------------------------
        for _, r in top_models.iterrows():
            mid = r["model_id"]
            clf_name = r["classifier"]
            pca_variant = r.get("pca", "no_pca")

            bp = r.get("best_params", "{}")
            if isinstance(bp, str):
                try:
                    bp = json.loads(bp)
                except Exception:
                    bp = {}
            if not isinstance(bp, dict):
                bp = {}

            gp = r.get("global_params", "{}")
            if isinstance(gp, str):
                try:
                    gp = json.loads(gp)
                except Exception:
                    gp = {}
            if not isinstance(gp, dict):
                gp = {}

            # coerce types (same helper you already use later)
            bp = {k: _coerce_param_value(v) for k, v in bp.items()}
            gp = {k: _coerce_param_value(v) for k, v in gp.items()}

            pipe, _ = build_pipeline(
                pca_variant=pca_variant,
                selector_name="NONE",
                clf_name=clf_name,
                fs_methods=None,
            )
            pipe.set_params(fsu="passthrough")

            if gp:
                pipe.set_params(**gp)
            if bp:
                pipe.set_params(**bp)

            pipe.fit(X_train, y_train)

            imp_df = top_features_with_scores_from_model(pipe, X_train, k=TOPK_PRINT)
            per_model_impdf[mid] = imp_df.copy()

            print("\n" + "-" * 90)
            print(f"[POST-HOC FS] {mid} ({clf_name}) TOP-{TOPK_PRINT} features + importance")
            print("-" * 90)

            if imp_df.empty:
                print("  (No importances available OR PCA active -> PCs, skipped.)")
                feats = []
            else:
                display(imp_df)
                # keep exactly TOP_FEATURES_PER_MODEL for voting
                feats = imp_df["feature"].tolist()[:TOP_FEATURES_PER_MODEL]

            per_model_feats[mid] = feats
            all_top_feats.extend(feats)

            print(f"[POST-HOC FS] {mid} kept_for_votes={len(feats)} (printed={min(TOPK_PRINT, len(imp_df))})")

        # ----------------------------
        # 2) Outputs dir + wide CSV (ONCE, after loop)
        # ----------------------------
        FINAL_OUT = os.path.join(OUT_DIR, "POSTHOC_IMPORTANCE_FS")
        os.makedirs(FINAL_OUT, exist_ok=True)

        WIDE_K = TOPK_PRINT
        wide_df = pd.DataFrame({"rank": list(range(1, WIDE_K + 1))})

        for mid, df in per_model_impdf.items():
            df2 = df.copy()
            if not df2.empty:
                df2 = df2[["feature", "importance"]].reset_index(drop=True)
            else:
                df2 = pd.DataFrame({"feature": [], "importance": []})

            if df2.shape[0] < WIDE_K:
                pad = pd.DataFrame({
                    "feature": ["" for _ in range(WIDE_K - df2.shape[0])],
                    "importance": [np.nan for _ in range(WIDE_K - df2.shape[0])],
                })
                df2 = pd.concat([df2, pad], ignore_index=True)
            else:
                df2 = df2.iloc[:WIDE_K].reset_index(drop=True)

            wide_df[f"{mid}__feature"] = df2["feature"].astype(str).values
            wide_df[f"{mid}__importance"] = df2["importance"].values

        wide_path = os.path.join(FINAL_OUT, f"top{WIDE_K}_CLASSIFIER_features_importance_wide.csv")
        wide_df.to_csv(wide_path, index=False)
        print("\n[POST-HOC FS] Saved wide top-features table:", wide_path)

        # ----------------------------
        # 3) Consensus across top models (vote-based)
        # ----------------------------
        feat_votes = Counter()
        for mid, feats in per_model_feats.items():
            feat_votes.update(set(feats))

        min_votes = int(POSTHOC_MIN_VOTES)
        consensus_feats = [f for f, v in feat_votes.items() if v >= min_votes]
        consensus_feats = sorted([f for f in consensus_feats if f in X_train.columns])

        union_feats = sorted(set(all_top_feats))
        union_feats = [f for f in union_feats if f in X_train.columns]

        if POSTHOC_MAX_KEEP is not None and len(consensus_feats) > int(POSTHOC_MAX_KEEP):
            consensus_feats = sorted(consensus_feats, key=lambda f: (-feat_votes[f], f))[: int(POSTHOC_MAX_KEEP)]
            consensus_feats = sorted(consensus_feats)

        print(f"\n[POST-HOC FS] Models used: {len(per_model_feats)}")
        print(f"[POST-HOC FS] Consensus rule: votes >= {min_votes}")
        print(f"[POST-HOC FS] Consensus features: {len(consensus_feats)}")

        final_feats = consensus_feats if len(consensus_feats) else union_feats
        if len(consensus_feats) == 0:
            print(f"[POST-HOC FS] WARNING: consensus empty → fallback to UNION ({len(union_feats)})")

        Xtr_fs = X_train[final_feats].copy()
        Xte_fs = X_test[final_feats].copy()

        print(f"[POST-HOC FS] Reduced DEV shape:  {Xtr_fs.shape}")
        print(f"[POST-HOC FS] Reduced TEST shape: {Xte_fs.shape}")

        # --- End-of-run OVERALL SANITY CHECK (includes selector + classifier feature counts) ---
        print_overall_sanity_from_results(
            results_df=results_df,
            final_refit_classifiers=FINAL_REFIT_CLASSIFIERS,
            final_feats_count=len(final_feats),
            posthoc_union_count=len(union_feats),
            posthoc_consensus_count=len(consensus_feats),
        )

        # ----------------------------
        # 4) Final refit models on reduced space
        # ----------------------------
        for final_clf in FINAL_REFIT_CLASSIFIERS:
            final_id = f"POSTHOC__TOP{TOP_IMPORTANCE_MODELS}x{TOP_FEATURES_PER_MODEL}__{final_clf}"
            print("\n" + "=" * 90)
            print(f"[FINAL REFIT] {final_id}")
            print("=" * 90)

            inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED)
            pipe, dist = build_final_refit_pipeline(final_clf, y_for_fit=y_train)

            search = make_search(pipe, dist, inner_cv)
            if search is None:
                best_est = pipe.fit(Xtr_fs, y_train)
                best_params = {}
            else:
                search.fit(Xtr_fs, y_train)
                best_est = search.best_estimator_
                best_params = search.best_params_

            tr_score = predict_score(best_est, Xtr_fs)
            te_score = predict_score(best_est, Xte_fs)

            thr = optimal_threshold_youden(y_train, tr_score)
            tr_m = compute_metrics(y_train, tr_score, threshold=thr)
            te_m = compute_metrics(y_test, te_score, threshold=thr)

            print("[FINAL REFIT] TRAIN:", _fmt_line(tr_m))
            print("[FINAL REFIT] TEST: ", _fmt_line(te_m))

            ddir = os.path.join(FINAL_OUT, final_id)
            os.makedirs(ddir, exist_ok=True)

            feat_vote_df = pd.DataFrame({
                "feature": final_feats,
                "votes": [int(feat_votes.get(f, 0)) for f in final_feats],
            })
            feat_vote_df.to_csv(os.path.join(ddir, "CLASSIFIER_selected_features_and_votes.csv"), index=False)

            pd.DataFrame({"patient_id": train_ids, "label": y_train, "score": tr_score}).to_csv(
                os.path.join(ddir, "train_predictions.csv"), index=False
            )
            pd.DataFrame({"patient_id": test_ids, "label": y_test, "score": te_score}).to_csv(
                os.path.join(ddir, "test_predictions.csv"), index=False
            )

            with open(os.path.join(ddir, "meta.json"), "w") as f:
                json.dump(
                    {
                        "final_id": final_id,
                        "top_importance_models": int(TOP_IMPORTANCE_MODELS),
                        "top_features_per_model": int(TOP_FEATURES_PER_MODEL),
                        "selected_models": top_models[["model_id", "classifier", "val_AUC_mean"]].to_dict(orient="records"),
                        "per_model_features": per_model_feats,
                        "union_feature_count": int(len(union_feats)),
                        "consensus_feature_count": int(len(consensus_feats)),
                        "best_params": best_params,
                        "thr_from_train": float(thr),
                        "train_metrics": tr_m,
                        "test_metrics": te_m,
                    },
                    f,
                    indent=2,
                )

        print("\n[POST-HOC FS] Done. Saved under:", FINAL_OUT)
# ============================================================
# NEW: PRINT END-OF-RUN TABLE FOR EACH TESTED PARAM COMBINATION
# (GLOBAL params + best_params + VAL + TEST metrics)
# ============================================================
cols = [
    "model_id",
    "classifier",
    "pca",
    "global_combo_index",
    "global_params",
    "best_params",
    "val_AUC_mean",
    "val_AUC_std",
    "val_ACC",
    "val_MCC",
    "val_YOUDEN",
    "test_AUC",
    "test_ACC",
    "test_MCC",
    "test_YOUDEN",
]

cols = [c for c in cols if c in results_df.columns]
end_table = results_df[cols].copy()

# Nice sorting: best validation mean AUC first, then stability, then MCC
sort_cols = [c for c in ["val_AUC_mean", "val_AUC_std", "val_MCC"] if c in end_table.columns]
if sort_cols:
    ascending_map = {
        "val_AUC_mean": False,
        "val_AUC_std": True,
        "val_MCC": False,
    }

    end_table = end_table.sort_values(
        sort_cols,
        ascending=[ascending_map[c] for c in sort_cols]
    )


print("\n" + "=" * 110)
print("END-OF-RUN: Metrics for EACH tested parameter combination (GLOBAL params + best_params)")
print("=" * 110)

# In notebooks, display a nice table
try:
    display(end_table.reset_index(drop=True))
except Exception:
    print(end_table.reset_index(drop=True).to_string(index=False))

# Also save it as a separate CSV for quick review
end_table_path = os.path.join(OUT_DIR, "all_param_combos_metrics.csv")
end_table.to_csv(end_table_path, index=False)
print("\nSaved per-parameter-combo table:", end_table_path)




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 22.3 MB/s eta 0:00:00
[UNION_FS] combos=1 | min_votes=3
[UNION_FS] selectors voting=6 | min_votes=3
Configured CLASSIFIERS: ['NAIVE_BAYES', 'GAUSSIAN_PROCESS', 'EXTRATREES', 'LOGREG', 'SGD_LOGLOSS']
[GLOBAL_GRID] combos=1 | AUC_FILTER=True | CORR_FILTER=True | keys=['aucf__high', 'aucf__low', 'corr__threshold']
Mounted at /content/drive
DEV (train) shape: (161, 138)  Pos%: 0.2422360248447205
TEST shape: (30, 138)  Pos%: 0.2

=== Running (1/5): no_pca__NAIVE_BAYES__G01__S01 ===
GLOBAL: {'aucf__high': 0.58, 'aucf__low': 0.42, 'corr__threshold': 0.98}
FS_COMBO: ('N_MRMR', 'mRMR', 'N_BORUTA', 'N_L1', 'N_ENET')
  Outer fold 1/5...
  [SHAP] fold 1: clf=NAIVE_BAYES | n_rows=138
  [SANITY] n_features after: var=137 | aucf=79 | corr=52 | fsu=26 | pca=None
  [SANITY-ID] fsu changed features: True
  [SANITY] features before FSU: 52  after FSU: 26
  [SANITY-ID] fsu-dropped features (first 20): ['Initial Volume [cm³]', 'grayROI3D_dbc_se', 'or

,fold,fold_thr_from_train,fold_val_auc,train_AUC,train_ACC,train_BACC,train_MCC,train_SENS,train_SPEC,train_YOUDEN,...,val_SPEC,val_YOUDEN,train_TN,train_FP,train_FN,train_TP,val_TN,val_FP,val_FN,val_TP
0,1,0.628652,0.690000,0.758231,0.781250,0.691054,0.391054,0.516129,0.865979,0.382108,...,0.760000,0.385000,84,13,15,16,19,6,3,5
1,2,0.943885,0.725714,0.804768,0.806202,0.724549,0.464625,0.562500,0.886598,0.449098,...,0.880000,0.308571,86,11,14,18,22,3,4,3
2,3,0.000842,0.593750,0.797893,0.751938,0.748519,0.441293,0.741935,0.755102,0.497038,...,0.583333,0.083333,74,24,8,23,14,10,4,4
3,4,0.005063,0.656250,0.798552,0.705426,0.739961,0.412914,0.806452,0.673469,0.479921,...,0.708333,0.333333,66,32,6,25,17,7,3,5
4,5,0.018435,0.640625,0.775181,0.744186,0.721363,0.399018,0.677419,0.765306,0.442725,...,0.750000,0.250000,75,23,10,21,18,6,4,4



[no_pca__NAIVE_BAYES__G01__S01] === FOLD SUMMARY (mean±std) ===


,metric,mean,std
0,fold_val_auc,0.661268,0.049972
1,train_AUC,0.786925,0.019590
2,train_ACC,0.757800,0.038260
3,train_BACC,0.725089,0.022038
4,train_MCC,0.421781,0.030648
5,train_SENS,0.660887,0.121105
6,train_SPEC,0.789291,0.087333
7,train_YOUDEN,0.450178,0.044076
8,train_PREC,0.515529,0.071477
9,train_RECALL,0.660887,0.121105



[no_pca__NAIVE_BAYES__G01__S01] === VALIDATION (OOF, fold-specific thresholds) ===
  AUC_OOF: 0.6462
  ACC_OOF: 0.6894
  BACC_OOF: 0.6381
  MCC_OOF: 0.2518
  YOUDEN_OOF: 0.2762
  TN_OOF: 90
  FP_OOF: 32
  FN_OOF: 18
  TP_OOF: 21
  AUC_FOLD_MEAN: 0.6613
  AUC_FOLD_STD: 0.0500
  PCA_N_MEAN: nan
  PCA_N_STD: nan

[no_pca__NAIVE_BAYES__G01__S01] === VALIDATION (OOF, ONE GLOBAL THRESHOLD from OOF) ===
  thr_oof=0.745998
  AUC=0.646 | ACC=0.752 | BACC=0.644 | MCC=0.300 | SENS=0.436 | SPEC=0.852 | YOUDEN=0.288 | PREC=0.486 | RECALL=0.436 | F1=0.459 | THRESH=0.746 | TN=104 | FP=18 | FN=22 | TP=17

[no_pca__NAIVE_BAYES__G01__S01] === TRAIN (final refit on full DEV, thr from DEV train) ===
  AUC=0.763 | ACC=0.683 | BACC=0.712 | MCC=0.366 | SENS=0.769 | SPEC=0.656 | YOUDEN=0.425 | PREC=0.417 | RECALL=0.769 | F1=0.541 | THRESH=0.071 | TN=80 | FP=42 | FN=9 | TP=30

[no_pca__NAIVE_BAYES__G01__S01] === TEST (holdout, evaluated at DEV train thr) ===
  AUC=0.833 | ACC=0.733 | BACC=0.771 | MCC=0.442 | 

...


Saved model-level metrics summary: /content/drive/MyDrive/NESTED_ML/results/EXP1/all_models_summary_metrics.csv
[FROZEN MODEL] Saved NAIVE_BAYES -> /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/NAIVE_BAYES__frozen_model.joblib
[FROZEN MODEL] Saved GAUSSIAN_PROCESS -> /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/GAUSSIAN_PROCESS__frozen_model.joblib
[FROZEN MODEL] Saved EXTRATREES -> /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/EXTRATREES__frozen_model.joblib
[FROZEN MODEL] Saved LOGREG -> /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/LOGREG__frozen_model.joblib
[FROZEN MODEL] Saved SGD_LOGLOSS -> /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/SGD_LOGLOSS__frozen_model.joblib
[FROZEN MODEL] Summary saved: /content/drive/MyDrive/NESTED_ML/results/EXP1/FROZEN_SINGLE_MODELS/frozen_models_summary.csv

[FROZEN MODEL] Final frozen single-classifier models:


,classifier,model_id,save_path,threshold,val_AUC_mean,val_AUC_std,val_MCC,best_params,global_params
0,NAIVE_BAYES,no_pca__NAIVE_BAYES__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.071054,0.661268,0.049972,0.251787,{},"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
1,GAUSSIAN_PROCESS,no_pca__GAUSSIAN_PROCESS__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.525475,0.529863,0.099455,-0.101222,{},"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
2,EXTRATREES,no_pca__EXTRATREES__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.416637,0.649583,0.073539,0.148902,"{""clf__max_depth"": 5, ""clf__max_features"": ""sq...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
3,LOGREG,no_pca__LOGREG__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,0.465771,0.671643,0.069564,0.198564,"{""clf__C"": 0.01, ""clf__max_iter"": 100, ""clf__t...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."
4,SGD_LOGLOSS,no_pca__SGD_LOGLOSS__G01__S01,/content/drive/MyDrive/NESTED_ML/results/EXP1/...,1.000000,0.471107,0.069122,-0.099046,"{""clf__alpha"": 1e-05, ""clf__class_weight"": nul...","{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_..."



[POST-HOC FS] Skipped (USE_POSTHOC_IMPORTANCE_FS=False).

END-OF-RUN: Metrics for EACH tested parameter combination (GLOBAL params + best_params)


,model_id,classifier,pca,global_combo_index,global_params,best_params,val_AUC_mean,val_AUC_std,val_ACC,val_MCC,val_YOUDEN,test_AUC,test_ACC,test_MCC,test_YOUDEN
0,no_pca__LOGREG__G01__S01,LOGREG,no_pca,1,"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_...","{""clf__C"": 0.01, ""clf__max_iter"": 100, ""clf__t...",0.671643,0.069564,0.602484,0.198564,0.231190,0.840278,0.733333,0.442269,0.541667
1,no_pca__NAIVE_BAYES__G01__S01,NAIVE_BAYES,no_pca,1,"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_...",{},0.661268,0.049972,0.689441,0.251787,0.276166,0.833333,0.733333,0.442269,0.541667
2,no_pca__EXTRATREES__G01__S01,EXTRATREES,no_pca,1,"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_...","{""clf__max_depth"": 5, ""clf__max_features"": ""sq...",0.649583,0.073539,0.695652,0.148902,0.144809,0.798611,0.733333,0.442269,0.541667
3,no_pca__GAUSSIAN_PROCESS__G01__S01,GAUSSIAN_PROCESS,no_pca,1,"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_...",{},0.529863,0.099455,0.726708,-0.101222,-0.040984,0.736111,0.800000,0.000000,0.000000
4,no_pca__SGD_LOGLOSS__G01__S01,SGD_LOGLOSS,no_pca,1,"{""aucf__high"": 0.58, ""aucf__low"": 0.42, ""corr_...","{""clf__alpha"": 1e-05, ""clf__class_weight"": nul...",0.471107,0.069122,0.527950,-0.099046,-0.111391,0.770833,0.700000,0.403604,0.500000



Saved per-parameter-combo table: /content/drive/MyDrive/NESTED_ML/results/EXP1/all_param_combos_metrics.csv
